<a href="https://colab.research.google.com/github/azkastudying1-hub/azkastudying1-hub/blob/main/Membangun_Proyek_Deep_Learning_Tingkat_Mahir.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Manipulasi Tensor dengan Operasi Dasar dan Matematika

In [ ]:
import tensorflow as tf

@tf.function
def my_function(x):
    tf.print("Inside function:", x)  # gunakan tf.print
    return x

print(my_function(tf.constant(5)))
print(my_function(tf.constant(10)))


In [ ]:
import tensorflow as tf

a = tf.Variable(3)
b = tf.constant(3)

a.assign(5)   # ✅ berhasil
b.assign(5)   # ❌ error, tidak ada method assign() -- Removed as tf.constant is immutable

print("Nilai a:", a)
print("Nilai b:", b)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
import tensorflow as tf

# 1. Membuat tensor 2D (Matrix) dengan nilai konstan
tensor1 = tf.constant([[1, 2, 3], [4, 5, 6]])

# 2. Mengakses elemen spesifik pada baris index 0 dan kolom index 0 (Angka 1)
elemen_pertama = tensor1[0, 0]

# 3. Mengambil seluruh elemen pada baris pertama (Index 0)
baris_pertama = tensor1[0]

# 4. Mengambil kolom pertama dari SEMUA baris (Tanda ':' berarti ambil semua)
kolom_pertama = tensor1[:, 0]

# Menampilkan hasil
print("Elemen Tunggal:", elemen_pertama.numpy())
print("Baris Pertama :", baris_pertama.numpy())
print("Kolom Pertama :", kolom_pertama.numpy())

In [ ]:
import tensorflow as tf

print(tf.constant(1, dtype=tf.int32))
print(tf.constant([1, 2, 3], dtype=tf.int32))
print(tf.constant([1, 4], dtype=tf.int32))

In [ ]:
tensor2 = tf.constant([9, 4, 2, 3, 7, 5, 6])

slicing = tf.slice(tensor2, begin=[1], size=[3])

print(slicing)

In [ ]:
tensor3 = tf.constant([1, 2, 3, 4])

reshaped_41 = tf.reshape(tensor3, (4, 1))
reshaped_22 = tf.reshape(tensor3, (2, 2))

In [ ]:
# Menampilkan Hasil
print("Original (1D):")
print(tensor3.numpy())

print("\nReshaped (4, 1):")
print(reshaped_41.numpy())

print("\nReshaped (2, 2):")
print(reshaped_22.numpy())

##Eksplorasi Tensor dalam Eager Mode

In [ ]:
import numpy as np
import tensorflow as tf

tensor1 = tf.constant(3)
tensor2 = tf.constant(8)

np.add(tensor1, tensor2)

In [ ]:
tensor3 = tf.constant([4, 3, 2, 1])
tensor3.numpy()

In [ ]:
array = np.array([4.0, 5.1, 6.0])
tensor = tf.convert_to_tensor(array)

##Graph Mode

In [ ]:
import tensorflow as tf
import time

def kalkulasi_rumit_eager(x):
  total = tf.constant(0.0)
  for i in tf.range(100000):
    total += tf.cast(i, dtype=tf.float32) * x
  return total


start_time = time.time()
hasil = kalkulasi_rumit_eager(tf.constant(2.0))
end_time = time.time()


print(f"Hasil Eager Mode: {hasil.numpy()}")
print(f"Waktu Eksekusi Eager Mode: {end_time - start_time:.4f} detik")

In [ ]:
@tf.function
def kalkulasi_rumit_graph(x):
  total = tf.constant(0.0)
  for i in tf.range(100000):
    total += tf.cast(i, dtype=tf.float32) * x
  return total

# Mengukur waktu eksekusi pertama kali
print("Eksekusi Pertama Graph (Kompilasi)")
start_time = time.time()
hasil = kalkulasi_rumit_graph(tf.constant(2.0))
end_time = time.time()

print(f"Hasil Graph Mode: {hasil.numpy()}")
print(f"Waktu Eksekusi Pertama Graph: {end_time - start_time:.4f} detik")

##Autograph dan TensorFlow Function

In [ ]:
def penjumlahan(a, b):
    return a + b

x = tf.constant(2)
y = tf.constant(5)

print(penjumlahan(x, y))

In [ ]:
@tf.function
def penjumlahan(a, b):
    return a + b

print(penjumlahan(x, y))

In [ ]:
def perkalian(a, b):
    return a * b

perkalian_graph = tf.function(perkalian)

In [ ]:
@tf.function
def penjumlahan(a, b):
    return a+b

In [ ]:
print(tf.autograph.to_code(penjumlahan.python_function))

In [ ]:
@tf.function
def add_numbers(a, b):
    print("Tracing terjadi!")
    return a + b

print(add_numbers(tf.constant(1), tf.constant(2)))
print(add_numbers(tf.constant(3), tf.constant(4)))

#Sequential API

#Latihan Custom Model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, LSTM
from tensorflow.keras.callbacks import Callback
from tensorflow.keras import Model

In [ ]:
# Unduh dataset dari Kaggle menggunakan KaggleHub
path = kagglehub.dataset_download("sumanthvrao/daily-climate-time-series-data")
file_name = 'DailyDelhiClimateTrain.csv'
file_path = os.path.join(path, file_name)

# Load dataset yang sudah diunduh dari Kaggle
df = pd.read_csv(file_path)

# Definisikan variabel dari dataset yang akan digunakan
temp = df['meantemp'].values

window_size = 60
batch_size = 100
shuffle_buffer = 1000


In [ ]:
class ResidualLSTM(tf.keras.layers.Layer):
    def __init__(self, units):
        super(ResidualLSTM, self).__init__()
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True)
        self.bn = tf.keras.layers.BatchNormalization()
        self.dense = tf.keras.layers.Dense(units, activation="relu")

    def call(self, inputs):
        x = self.lstm(inputs)
        x = self.bn(x)
        residual = self.dense(inputs)
        return x + residual

In [ ]:
class ResidualLSTMModel(tf.keras.Model):
    def __init__(self):
        super(ResidualLSTMModel, self).__init__()
        # LSTM pertama tetap return_sequences=True untuk diumpankan ke LSTM kedua
        self.residual_lstm1 = ResidualLSTM(60)
        # LSTM kedua diubah agar tidak return_sequences (hanya output terakhir)
        # atau kita tambahkan Flatten/GlobalPooling sebelum Dense layer.
        self.residual_lstm2 = tf.keras.layers.LSTM(60, return_sequences=False)
        self.dense1 = tf.keras.layers.Dense(30, activation="relu")
        self.dense2 = tf.keras.layers.Dense(10, activation="relu")
        self.dense3 = tf.keras.layers.Dense(1)

    def call(self, inputs):
        x = self.residual_lstm1(inputs)
        x = self.residual_lstm2(x)
        x = self.dense1(x)
        x = self.dense2(x)
        return self.dense3(x)

In [ ]:
class CustomHuberLoss(tf.keras.losses.Loss):
    def __init__(self, delta=1.0):
        super(CustomHuberLoss, self).__init__()
        self.delta = delta

    def call(self, y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) <= self.delta
        squared_loss = tf.square(error) / 2
        linear_loss = self.delta * (tf.abs(error) - self.delta / 2)
        return tf.where(is_small_error, squared_loss, linear_loss)

In [ ]:
class AdaptiveLearningRateScheduler(Callback):
    def __init__(self, factor=0.5, patience=3, min_lr=1e-6, max_lr=1e-2):
        super(AdaptiveLearningRateScheduler, self).__init__()
        self.factor = factor
        self.patience = patience
        self.min_lr = min_lr
        self.max_lr = max_lr
        self.wait = 0
        self.best_loss = float('inf')

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get("loss")
        lr = tf.keras.backend.get_value(self.model.optimizer.learning_rate)

        if current_loss < self.best_loss:
            pass # Added to resolve the SyntaxError. Replace with actual logic.

In [ ]:
class AdaptiveLearningRateScheduler(Callback):
    def __init__(self, factor=0.5, patience=3, min_lr=1e-6, max_lr=1e-2):
        super(AdaptiveLearningRateScheduler, self).__init__()
        self.factor = factor
        self.patience = patience
        self.min_lr = min_lr
        self.max_lr = max_lr
        self.wait = 0
        self.best_loss = float('inf')

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get("loss")
        # Perbaikan akses LR untuk Keras 3
        lr = float(self.model.optimizer.learning_rate.numpy() if hasattr(self.model.optimizer.learning_rate, 'numpy') else self.model.optimizer.learning_rate)

        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                new_lr = max(lr * self.factor, self.min_lr)
                # Perbaikan cara set_value yang menyebabkan AttributeError
                self.model.optimizer.learning_rate.assign(new_lr)
                self.wait = 0
                print(f"\nEpoch {epoch+1}: Reducing learning rate to {new_lr}.")

In [ ]:
# Update windowed_dataset dengan .repeat() untuk mencegah data habis
def windowed_dataset(series, window_size, batch_size, shuffle_buffer):
    dataset = tf.data.Dataset.from_tensor_slices(series)
    dataset = dataset.window(window_size + 1, shift=1, drop_remainder=True)
    dataset = dataset.flat_map(lambda w: w.batch(window_size + 1))
    dataset = dataset.shuffle(shuffle_buffer)
    dataset = dataset.map(lambda w: (tf.expand_dims(w[:-1], axis=-1), w[-1:]))
    # Tambahkan repeat agar dataset tidak habis sebelum epoch selesai
    return dataset.batch(batch_size).repeat().prefetch(1)

train_set = windowed_dataset(temp, window_size, batch_size, shuffle_buffer)

optimizer = tf.keras.optimizers.SGD(learning_rate=1.0000e-04, momentum=0.9)

model = ResidualLSTMModel()
model.compile(loss=CustomHuberLoss(delta=0.5), optimizer=optimizer)

lr_callback = AdaptiveLearningRateScheduler()
# Tambahkan steps_per_epoch karena dataset sekarang bersifat infinite (repeat)
steps = (len(temp) - window_size) // batch_size
model.fit(train_set, epochs=100, steps_per_epoch=steps, callbacks=[lr_callback])

In [ ]:
# Contoh alternatif menggunakan optimizer Adam dan metrik tambahan
new_model = ResidualLSTMModel()

new_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=CustomHuberLoss(delta=0.5),
    metrics=['mae'] # Menambahkan Mean Absolute Error sebagai metrik
)

# Menjalankan pelatihan
history = new_model.fit(
    train_set,
    epochs=50,
    steps_per_epoch=steps,
    callbacks=[lr_callback]
)

In [ ]:
# Visualisasi hasil pelatihan
plt.plot(history.history['loss'], label='Loss')
plt.plot(history.history['mae'], label='MAE')
plt.title('Model Training Progress')
plt.xlabel('Epoch')
plt.ylabel('Value')
plt.legend()
plt.show()

#Latihan: Custom dan Dynamic Training

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time

# Muat dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalisasi dan tambahkan channel dimension
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = x_train[..., tf.newaxis]
x_test = x_test[..., tf.newaxis]

# Dataset pipeline
batch_size = 64
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(buffer_size=1024).batch(batch_size)
val_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(batch_size)

In [ ]:
class MyModel(tf.keras.Model):
    def __init__(self):
        super(MyModel, self).__init__()
        # Pastikan layer didefinisikan sebagai atribut self
        self.flatten = tf.keras.layers.Flatten()
        self.dense1 = tf.keras.layers.Dense(128, activation='relu')
        self.dense2 = tf.keras.layers.Dense(10)

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.dense1(x)
        return self.dense2(x)

In [ ]:
# Loss function untuk klasifikasi multi-class dengan label integer
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Optimizer Adam untuk memperbarui bobot model
optimizer = tf.keras.optimizers.Adam()

# Metrik untuk memantau performa training & validasi
train_loss_metric = tf.keras.metrics.Mean(name='train_loss')                     # Rata-rata loss selama training
train_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')  # Akurasi training
val_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')      # Akurasi validasi


##Custom Training

In [ ]:
# Fungsi training step (satu batch data)
@tf.function
def train_step(model, images, labels):
    with tf.GradientTape() as tape:
        predictions = model(images, training=True)
        loss = loss_object(labels, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    train_loss_metric.update_state(loss)
    train_accuracy_metric.update_state(labels, predictions)

# Fungsi validasi step (satu batch data)
@tf.function
def val_step(model, images, labels):
    predictions = model(images, training=False)
    val_accuracy_metric.update_state(labels, predictions)

In [ ]:
# Fungsi utama untuk melatih model
def train_model(model, train_ds, val_ds, epochs=10):
    for epoch in range(epochs):
        start = time.time()
        print(f"\nEpoch {epoch+1}/{epochs}")

        train_loss_metric.reset_state()
        train_accuracy_metric.reset_state()
        val_accuracy_metric.reset_state()

        for images, labels in train_ds:
            train_step(model, images, labels)

        for val_images, val_labels in val_ds:
            val_step(model, val_images, val_labels)

        current_train_loss = train_loss_metric.result()
        current_train_accuracy = train_accuracy_metric.result() * 100
        current_val_accuracy = val_accuracy_metric.result() * 100

        print(
            f"Train Accuracy: {current_train_accuracy:.2f}%, ",
            f"Val Accuracy: {current_val_accuracy:.2f}%",
            f"({time.time() - start:.2f}s)"
        )
        # Log informasi ke file
        logging.info(f"Epoch {epoch+1}/{epochs}, Train Loss: {current_train_loss:.4f}, Train Accuracy: {current_train_accuracy:.2f}%, Val Accuracy: {current_val_accuracy:.2f}%")

In [ ]:
model = MyModel()

# Re-initialize optimizer and metrics for the new model instance
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()
train_loss_metric = tf.keras.metrics.Mean(name='train_loss')
train_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
val_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

train_model(model, train_ds, val_ds)

In [ ]:
import logging

# Setup logger: simpan log ke file 'training.log' dengan level INFO
logging.basicConfig(filename='training.log', level=logging.INFO)

In [ ]:
# Ambil satu batch data dari dataset
for x, y in train_ds.take(1):
    break

# Gunakan loss_object yang sudah didefinisikan sebelumnya
loss_fn = loss_object

# Ambil bobot awal sebelum diupdate
initial_weights = model.get_weights()

# Forward pass dan hitung loss
with tf.GradientTape() as tape:
    y_pred = model(x, training=True)
    loss_value = loss_fn(y, y_pred)

# Deteksi NaN atau Inf pada loss
tf.debugging.check_numerics(loss_value, "Loss has NaN or Inf")

# Komputasi gradien dan update bobot
gradients = tape.gradient(loss_value, model.trainable_variables)
optimizer.apply_gradients(zip(gradients, model.trainable_variables))

# Ambil bobot setelah diupdate
final_weights = model.get_weights()

print("Successfully executed one manual training step.")
print(f"Loss value: {loss_value.numpy():.4f}")
print("Weights have been captured in 'initial_weights' and 'final_weights'.")

##Dynamic Training

In [ ]:
def dynamic_train_model(model, train_ds, val_ds, epochs=10, stagnation_threshold=3, checkpoint_manager=None):
    best_val_acc = 0.0  # Menyimpan akurasi validasi terbaik sebagai float
    stagnation_counter = 0

    for epoch in range(epochs):
        start = time.time()

        # Reset metrik di awal setiap epoch
        train_loss_metric.reset_state()
        train_accuracy_metric.reset_state()
        val_accuracy_metric.reset_state()

        # Loop Training
        for images, labels in train_ds:
            train_step(model, images, labels)

        # Loop Validasi
        for val_images, val_labels in val_ds:
            val_step(model, val_images, val_labels)

        # Ambil hasil akurasi validasi
        val_acc = val_accuracy_metric.result().numpy()
        print(f"Epoch {epoch+1}, val_acc: {val_acc:.4f} (Time: {time.time() - start:.2f}s)")

        # Simpan hanya jika val_acc meningkat sesuai konsep yang diminta
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            stagnation_counter = 0
            if checkpoint_manager:
                checkpoint_manager.save()
                print("✅ Checkpoint disimpan karena val_acc meningkat.")
        else:
            stagnation_counter += 1

        # Early stopping jika tidak ada peningkatan
        if stagnation_counter >= stagnation_threshold:
            print("⚠️ Performa stagnan, training dihentikan lebih awal!")
            break

In [ ]:
model = MyModel()

# Re-inisialisasi variabel pendukung
optimizer = tf.keras.optimizers.Adam()
train_loss_metric = tf.keras.metrics.Mean(name='train_loss')
train_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
val_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

# Force building model agar variabel terdeteksi oleh optimizer
dummy_input = tf.ones((1, 28, 28, 1))
model(dummy_input)
optimizer.build(model.trainable_variables)

# Jalankan training dengan logika penyimpanan kondisional baru
dynamic_train_model(model, train_ds, val_ds, epochs=10, checkpoint_manager=checkpoint_manager)

In [ ]:
import os

# Membuat objek checkpoint untuk model dan optimizer
checkpoint = tf.train.Checkpoint(model=model, optimizer=optimizer)

# Tentukan direktori penyimpanan
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")

# Menambahkan CheckpointManager untuk mengelola rotasi file checkpoint
checkpoint_manager = tf.train.CheckpointManager(
    checkpoint, directory='./checkpoints', max_to_keep=3)

print("Checkpoint and CheckpointManager created successfully.")

In [ ]:
checkpoint = tf.train.Checkpoint(optimizer=optimizer, model=model)
manager = tf.train.CheckpointManager(checkpoint, './checkpoints', max_to_keep=3)

# Cek apakah ada checkpoint yang tersedia
if manager.latest_checkpoint:
    checkpoint.restore(manager.latest_checkpoint)
    print("Checkpoint berhasil dimuat dari:", manager.latest_checkpoint)
else:
    print("Tidak ditemukan checkpoint, training akan dimulai dari awal.")

In [ ]:
# Verifikasi akhir summary model dan status checkpoint
model.summary()

print(f"\nCheckpoint status:")
print(f"Model layers built: {model.built}")
print(f"Optimizer built: {optimizer.built}")
print(f"Checkpoint prefix path: {checkpoint_prefix}")

In [ ]:
# Menampilkan checkpoint terbaru yang tersedia
latest_ckpt = checkpoint_manager.latest_checkpoint
print(f"Latest checkpoint: {latest_ckpt}")

# Restore model dan optimizer ke state terakhir
# .expect_partial() digunakan jika kita tidak ingin mengecek setiap variabel (seperti internal Keras metrics)
status = checkpoint.restore(latest_ckpt).expect_partial()

print("Model and optimizer restored successfully.")

# Verifikasi dengan mengecek akurasi pada data validasi menggunakan model yang direstore
val_accuracy_metric.reset_state()
for images, labels in val_ds:
    val_step(model, images, labels)

print(f"Restored Model Val Accuracy: {val_accuracy_metric.result() * 100:.2f}%")

In [ ]:
# Menampilkan checkpoint terbaru yang tersedia
latest_ckpt = checkpoint_manager.latest_checkpoint
print(f"Latest checkpoint: {latest_ckpt}")

# Restore model dan optimizer ke state terakhir
# .expect_partial() digunakan jika kita tidak ingin mengecek setiap variabel (seperti internal Keras metrics)
status = checkpoint.restore(latest_ckpt).expect_partial()

print("Model and optimizer restored successfully.")

# Verifikasi dengan mengecek akurasi pada data validasi menggunakan model yang direstore
val_accuracy_metric.reset_state()
for images, labels in val_ds:
    val_step(model, images, labels)

print(f"Restored Model Val Accuracy: {val_accuracy_metric.result() * 100:.2f}%")

#Latihan Hybrid Recommendation

In [ ]:
# Import library utama untuk data science & machine learning
import pandas as pd                     # Manipulasi data tabular
import numpy as np                      # Operasi numerik dan array
import kagglehub                        # Akses dataset dari Kaggle
import tensorflow as tf                 # Framework deep learning
from tensorflow.keras.metrics import MeanSquaredError   # Metrik regresi (MSE)

# Preprocessing dari scikit-learn
from sklearn.preprocessing import (
    MultiLabelBinarizer,   # Konversi label multi-kategori ke bentuk biner
    OneHotEncoder,         # Encoding kategori ke vektor one-hot
    StandardScaler,        # Normalisasi fitur (mean=0, std=1)
    LabelEncoder           # Encoding label kategori ke integer
)

# Evaluasi & transformasi data
from sklearn.metrics.pairwise import cosine_similarity   # Mengukur kesamaan antar vektor
from sklearn.model_selection import train_test_split     # Membagi dataset ke train & test
from sklearn.decomposition import PCA                    # Reduksi dimensi dengan PCA

# Visualisasi
import matplotlib.pyplot as plt          # Plot grafik dan visualisasi data


In [ ]:
# Unduh dataset Anime Recommendations dari Kaggle
path = kagglehub.dataset_download("CooperUnion/anime-recommendations-database")

# Cetak lokasi folder hasil unduhan
print("Path to dataset files:", path)


In [ ]:
df_content = pd.read_csv(f"{path}/anime.csv")
df_rating = pd.read_csv(f"{path}/rating.csv")

##Persiapan Dataset dan Dependencies

##Eksplorasi Data

In [ ]:
df_content.head()

In [ ]:
df_rating.head()

##Preprocessing Data Item-based

In [ ]:
rating_counts = df_rating['rating'].value_counts()

plt.figure(figsize=(8, 4))
rating_counts.plot(kind='bar', color='skyblue')
plt.title('Distribusi Rating Sebelum Pembersihan')
plt.xlabel('Rating')
plt.ylabel('Jumlah')
plt.show()

In [ ]:
df_rating_cleaned = df_rating[df_rating['rating'] != -1]
rating_cleaned_counts = df_rating_cleaned['rating'].value_counts()

plt.figure(figsize=(8, 4))
rating_cleaned_counts.plot(kind='bar', color='skyblue')
plt.title('Distribusi Rating Setelah Pembersihan')
plt.xlabel('Rating')
plt.ylabel('Jumlah')
plt.show()

###Mengubah Fitur Menjadi Representasi Vektor

In [ ]:
genre = "Action, Adventure, Fantasy"

In [ ]:
mlb_genre = MultiLabelBinarizer()

df_content['genre'] = df_content['genre'].astype(str).fillna('')
df_content['genre_list'] = df_content['genre'].str.split(', ')
X_genre = mlb_genre.fit_transform(df_content['genre_list'])

In [ ]:
enc_type = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_type = enc_type.fit_transform(df_content[['type']])

###Normalisasi Fitur Numerik

In [ ]:
num_cols = ['episodes', 'rating', 'members']
scaler = StandardScaler()
for col in num_cols:
    df_content[col] = pd.to_numeric(df_content[col], errors='coerce').fillna(0)
X_num = scaler.fit_transform(df_content[num_cols].fillna(0))

###Menggabungkan Fitur

In [ ]:
content_features = np.hstack([X_genre, X_type, X_num])
content_features /= np.linalg.norm(content_features, axis=1, keepdims=True)

##Preprocessing Data User-based

###Menyiapkan Data untuk NCF

In [ ]:
# Collaborative Filtering menggunakan Neural CF
# Persiapkan pasangan user-item untuk training model
encode_user = LabelEncoder()
encode_item = LabelEncoder()

df_rating_cleaned['user_idx'] = encode_user.fit_transform(df_rating_cleaned['user_id'])
df_rating_cleaned['item_idx'] = encode_item.fit_transform(df_rating_cleaned['anime_id'])
num_users = df_rating_cleaned['user_idx'].nunique()
num_items = df_rating_cleaned['item_idx'].nunique()

###Membagi Data Latih dan Validasi

In [ ]:
df_train, df_val = train_test_split(df_rating_cleaned, test_size=0.2, random_state=42)

###Membuat Pipeline tf.data

In [ ]:
# Bangun tf.data pipeline
train_ds = tf.data.Dataset.from_tensor_slices((
    {
        'user_id': df_train['user_idx'].values,
        'item_id': df_train['item_idx'].values
    },
    df_train['rating'].astype('float32').values
))

val_ds = tf.data.Dataset.from_tensor_slices((
    {
        'user_id': df_val['user_idx'].values,
        'item_id': df_val['item_idx'].values
    },
    df_val['rating'].astype('float32').values
))

train_ds = train_ds.shuffle(1000000).batch(512).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(512).prefetch(tf.data.AUTOTUNE)

##Definisi Model NCF dan Pelatihan

###Membangun Model NCF

In [ ]:
# Input Layer
def build_inputs():
    u_in = tf.keras.Input(shape=(), name='user_id', dtype='int32')
    i_in = tf.keras.Input(shape=(), name='item_id', dtype='int32')

    return u_in, i_in

In [ ]:
# GMF (Generalized Matrix Factorization) branch
def build_gmf_branch(u_in, i_in, num_users, num_items, emb_dim):
    u_g = tf.keras.layers.Embedding(num_users, emb_dim, name='gmf_user_emb')(u_in)
    i_g = tf.keras.layers.Embedding(num_items, emb_dim, name='gmf_item_emb')(i_in)
    u_g = tf.keras.layers.Flatten()(u_g)
    i_g = tf.keras.layers.Flatten()(i_g)
    gmf_out = tf.keras.layers.Multiply()([u_g, i_g])  # element-wise product

    return gmf_out

In [ ]:
# MLP branch
def build_mlp_branch(u_in, i_in, num_users, num_items, emb_dim):
    u_m = tf.keras.layers.Embedding(num_users, emb_dim, name='mlp_user_emb')(u_in)
    i_m = tf.keras.layers.Embedding(num_items, emb_dim, name='mlp_item_emb')(i_in)
    u_m = tf.keras.layers.Flatten()(u_m)
    i_m = tf.keras.layers.Flatten()(i_m)
    mlp_out = tf.keras.layers.Concatenate()([u_m, i_m])
    mlp_out = tf.keras.layers.Dense(64, activation='relu')(mlp_out)
    mlp_out = tf.keras.layers.Dropout(0.3)(mlp_out)
    mlp_out = tf.keras.layers.Dense(32, activation='relu')(mlp_out)

    return mlp_out

In [ ]:
# Gabungkan GMF + MLP menjadi NeuMF
def build_ncf(num_users, num_items, emb_dim=64):
    u_in, i_in = build_inputs()

    gmf_out = build_gmf_branch(u_in, i_in, num_users, num_items, emb_dim)
    mlp_out = build_mlp_branch(u_in, i_in, num_users, num_items, emb_dim)

    fusion = tf.keras.layers.Concatenate()([gmf_out, mlp_out])
    output = tf.keras.layers.Dense(1, activation='linear', name='prediction')(fusion)

    model = tf.keras.Model(inputs=[u_in, i_in], outputs=output)
    model.compile(optimizer='adam', loss='mse')
    return model

ncf_model = build_ncf(num_users, num_items, emb_dim=16)
ncf_model.summary()

In [ ]:
ncf_model.fit(train_ds, epochs=5, validation_data=val_ds, verbose=1)

##Ekstraksi Hasil Embedding

In [ ]:
# Ambil input item dari model (biasanya indeks item)
item_input = ncf_model.input[1]   # i_in

# Ambil layer embedding item dari GMF dan MLP
gmf_item_emb_layer = ncf_model.get_layer('gmf_item_emb')
mlp_item_emb_layer = ncf_model.get_layer('mlp_item_emb')

# Flatten hasil embedding agar berbentuk vektor 1D
gmf_item_emb = tf.keras.layers.Flatten()(gmf_item_emb_layer(item_input))
mlp_item_emb = tf.keras.layers.Flatten()(mlp_item_emb_layer(item_input))

# Gabungkan embedding GMF dan MLP untuk feature augmentation
combined_item_emb = tf.keras.layers.Concatenate()([gmf_item_emb, mlp_item_emb])


In [ ]:
import tensorflow as tf
import numpy as np

# Ambil input item dari model (biasanya indeks item)
# `ncf_model` and `num_items` are expected to be defined in previous cells.
item_input = ncf_model.input[1]   # i_in

# Ambil layer embedding item dari GMF dan MLP
gmf_item_emb_layer = ncf_model.get_layer('gmf_item_emb')
mlp_item_emb_layer = ncf_model.get_layer('mlp_item_emb')

# Flatten hasil embedding agar berbentuk vektor 1D
gmf_item_emb = tf.keras.layers.Flatten()(gmf_item_emb_layer(item_input))
mlp_item_emb = tf.keras.layers.Flatten()(mlp_item_emb_layer(item_input))

# Gabungkan embedding GMF dan MLP untuk feature augmentation
combined_item_emb = tf.keras.layers.Concatenate()([gmf_item_emb, mlp_item_emb])

# Buat model khusus untuk mengekstrak embedding item gabungan (GMF + MLP)
emb_model = tf.keras.Model(inputs=item_input, outputs=combined_item_emb)

# Ekstraksi embedding untuk semua item
all_items = np.arange(num_items, dtype=np.int32)              # Buat array indeks item
item_embs = emb_model.predict(all_items, batch_size=512)      # Prediksi embedding
item_embs = item_embs.squeeze()                               # Hilangkan dimensi ekstra

# Normalisasi embedding agar setiap vektor punya panjang 1 (unit vector)
item_embs /= np.linalg.norm(item_embs, axis=1, keepdims=True)

##Feature Augmentation

In [ ]:
# Feature Augmentation
# Cari data anime yang ada di kedua sumber (content + CF)
common_mask = df_content['anime_id'].isin(encode_item.classes_)   # Mask anime yang cocok
filtered_idx = np.where(common_mask.values)[0]                    # Ambil indeks konten yang valid

# Ambil indeks sesuai versi CF dan cocokkan dengan fitur konten
cf_indices = encode_item.transform(df_content.loc[common_mask, 'anime_id'])
content_f = content_features[filtered_idx]   # Fitur konten (misalnya genre, studio)
cf_f = item_embs[cf_indices]                 # Fitur CF (embedding dari NCF)

# Gabungkan kedua jenis fitur jadi satu representasi
aug_features = np.hstack([content_f, cf_f])

# Normalisasi supaya setiap vektor punya panjang yang sama (unit vector)
aug_features /= np.linalg.norm(aug_features, axis=1, keepdims=True)


##Rekomendasi dengan Similarity

In [ ]:
# Perhitungan similarity antar anime dengan fitur augmented
sim_aug = cosine_similarity(aug_features)

# Metadata anime yang valid (ada di kedua sumber)
idx_meta = df_content.loc[common_mask].reset_index(drop=True)

# Fungsi inference rekomendasi
def recommend(anime_id, top_n=10):
    # Cari indeks anime berdasarkan ID
    idx = idx_meta.index[idx_meta['anime_id'] == anime_id][0]

    # Ambil similarity vector untuk anime tersebut
    sims = sim_aug[idx]

    # Urutkan dari yang paling mirip (descending)
    order = np.argsort(sims)[::-1]

    # Buang diri sendiri (anime yang sama)
    order = order[order != idx]

    # Ambil top-N anime mirip
    top = order[:top_n]

    # Return daftar rekomendasi (anime_id, nama, skor similarity)
    return [
        (int(idx_meta.iloc[i]['anime_id']),
         idx_meta.iloc[i]['name'],
         float(sims[i]))
        for i in top
    ]


#Latihan Data Labeling

In [ ]:
!pip install --upgrade transformers tf-keras

In [ ]:
!pip install emoji


In [ ]:
# Explicitly set environment variables for Hugging Face Transformers to use TensorFlow
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' # Sometimes helps with specific TensorFlow issues
os.environ['TRANSFORMERS_BACKEND'] = 'tf' # Explicitly set backend to TensorFlow

In [ ]:
!pip install --upgrade transformers

In [ ]:
import os
# Memaksa TensorFlow untuk menggunakan Keras backend versi lama (Keras 2)
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Install ulang package yang kompatibel
!pip install --upgrade "transformers>=4.40.0" "tf-keras"

In [ ]:
# WAJIB DI BARIS PALING ATAS SEBELUM TENSORFLOW DIMUAT
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Install dependensi pendukung secara diam-diam
# (Cukup jalankan sekali saja, setelah itu baris pip ini bisa dihapus/dikomen)
!pip install -q tf-keras transformers

import tensorflow as tf

@tf.function
def my_function(x):
    tf.print("Inside function:", x)
    return x

print(my_function(tf.constant(5)))
print(my_function(tf.constant(10)))

In [ ]:
# =========================================================
# 1. SET ENVIRONMENT (Wajib di baris pertama sebelum import)
# =========================================================
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# =========================================================
# 2. IMPORT LIBRARY UTAMA & TENSORFLOW
# =========================================================
import pandas as pd
import numpy as np
import re
import string
import html
import emoji
import tensorflow as tf

# =========================================================
# 3. IMPORT HUGGING FACE TRANSFORMERS
# =========================================================
# Pastikan lingkungan 'transformers' mengetahui untuk menggunakan backend TensorFlow
os.environ['TRANSFORMERS_BACKEND'] = 'tf'

from transformers import (
    AutoTokenizer, # Use AutoTokenizer for flexibility
    AutoModelForSequenceClassification # Use AutoModelForSequenceClassification instead of TFAutoModelForSequenceClassification
)

print("✅ Semua library dan model BERT berhasil dimuat tanpa error!")

In [ ]:
import kagglehub

# Download dataset menggunakan kagglehub
path = kagglehub.dataset_download("khalidryder777/500k-chatgpt-tweets-jan-mar-2023")

In [ ]:
csv_name = 'Twitter Jan Mar.csv'
file_path = os.path.join(path, csv_name)

df = pd.read_csv(file_path)
df

In [ ]:
df = df[["content"]]
df = df.head(500)

df

##Preprocessing Data

In [ ]:
# Cleaning Data
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)
    text = re.sub(r"\d+", "", text)
    text = emoji.replace_emoji(text, replace='')
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()

    return text.lower()

df["content_clean"] = df["content"].apply(clean_text)
df.head()

##Inisialisasi Model dan Tokenizer BERT

In [ ]:
model_name = 'nlptown/bert-base-multilingual-uncased-sentiment'

# Inisialisasi Model dan Tokenizer BERT for sentiment
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

##Prediksi Label pada Data

In [ ]:
from transformers import pipeline
import pandas as pd

# Definisikan model name
model_name = 'nlptown/bert-base-multilingual-uncased-sentiment'

# Inisialisasi sentiment analysis pipeline
# Pipeline ini secara otomatis menangani tokenizer dan model
print("Memuat model ke dalam pipeline...")
sentiment_pipeline = pipeline("sentiment-analysis", model=model_name)

# Persiapkan list untuk menyimpan hasil
predictions = []

print("Memulai proses labeling pada 500 data...")
# Melakukan prediksi dalam batch agar lebih cepat
for i in range(0, len(df)):
    text = df["content_clean"].iloc[i]
    if text.strip() == "":
        predictions.append("3 stars") # Default untuk teks kosong
        continue

    # Prediksi menggunakan pipeline
    result = sentiment_pipeline(text[:512]) # Batasi panjang karakter
    predictions.append(result[0]['label'])

# Simpan hasil ke dataframe
df['sentiment'] = predictions
print("✅ Labeling selesai!")
print(df[['content_clean', 'sentiment']].head())

In [ ]:
# Tambahkan hasil prediksi ke dataframe
df["predicted_label"] = predictions

# Mapping label awal menjadi 3 kategori (Positive, Neutral, Negative)
def map_rating_to_sentiment(label):
    if label in ["1 star", "2 stars"]:
        return "negative"
    elif label == "3 stars":
        return "neutral"
    else:
        return "positive"

# Terapkan mapping untuk membuat kolom sentiment yang lebih sederhana
df["sentiment_category"] = df["predicted_label"].apply(map_rating_to_sentiment)

print("✅ Berhasil memetakan rating ke kategori sentimen!")
print(df[['content_clean', 'predicted_label', 'sentiment_category']].head())

In [ ]:
df

In [ ]:
pd.set_option('display.max_colwidth', None)

# Filter menggunakan kolom 'sentiment_category' yang benar
positive_tweets = df[df['sentiment_category'] == 'positive']
negative_tweets = df[df['sentiment_category'] == 'negative']
neutral_tweets = df[df['sentiment_category'] == 'neutral']

print("Contoh Tweet Positif:")
display(positive_tweets[['content', 'sentiment_category']].sample(min(5, len(positive_tweets))))

print("\nContoh Tweet Negatif:")
display(negative_tweets[['content', 'sentiment_category']].sample(min(5, len(negative_tweets))))

print("\nContoh Tweet Netral:")
display(neutral_tweets[['content', 'sentiment_category']].sample(min(5, len(neutral_tweets))))

##Simpan Data yang Sudah Dilabeli

In [ ]:
df.to_csv("data_with_sentiment.csv", index=False)

#Latihan Membangun Model Transformer Milik Kita Sendiri dengan TensorFlow

##Persiapan Dependencies

In [ ]:
!pip install tensorflow==2.19.0
!pip install tensorflow_datasets==4.9.9
!pip install transformers==4.55.2

In [ ]:
# Import library utama untuk data science & deep learning
import numpy as np                      # Operasi numerik dan array
import tensorflow as tf                 # Framework machine learning
from tensorflow import keras            # API Keras di dalam TensorFlow
from tensorflow.keras import layers     # Layer-layer untuk membangun model

# Dataset bawaan TensorFlow
import tensorflow_datasets as tfds      # Koleksi dataset siap pakai

# Visualisasi
import matplotlib.pyplot as plt         # Plot grafik dan visualisasi hasil

# Library Hugging Face untuk NLP
import transformers                     # Framework NLP berbasis Transformer
from transformers import AutoTokenizer  # Tokenizer otomatis untuk berbagai model Transformer


##Embedding


In [ ]:
def positional_encoding(length, depth):
    # Membagi dua dimensi untuk sin & cos
    depth = depth / 2

    # Array posisi dengan shape (length, 1)
    positions = np.arange(length)[:, np.newaxis]

    # Array dimensi dengan shape (1, depth)
    depths = np.arange(depth)[np.newaxis, :] / depth

    # Perhitungan sudut untuk sinus & cosinus
    angle_rates = 1 / (10000**depths)
    angle_rads = positions * angle_rates

    # Hitung nilai sinus dan cosinus
    pos_encoding = np.concatenate(
        [np.sin(angle_rads), np.cos(angle_rads)],
        axis=-1
    )

    # Konversi ke tensor float32 dengan shape (1, length, depth)
    return tf.cast(pos_encoding, dtype=tf.float32)[tf.newaxis, ...]


In [ ]:
class Positional_Embedding(layers.Layer):
    def __init__(self, vocab_size, d_model, max_length=2048):
        super().__init__()
        # Dimensi embedding
        self.d_model = d_model
        # Layer embedding untuk token input
        self.embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=d_model,
            mask_zero=True
        )
        # Positional encoding (sinus-cosinus) untuk sequence
        self.pos_encoding = positional_encoding(
            length=max_length,
            depth=d_model
        )

    # Fungsi untuk meneruskan masking (berguna di decoder)
    def compute_mask(self, *args, **kwargs):
        return self.embedding.compute_mask(*args, **kwargs)

    def call(self, x):
        # Panjang sequence input
        length = tf.shape(x)[1]
        # Embedding token → (batch_size, seq_length, d_model)
        x = self.embedding(x)
        # Skala embedding agar stabil
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        # Tambahkan positional encoding sesuai panjang sequence
        x = x + self.pos_encoding[:, :length, :]
        return x


##Multi-Head Attention

In [ ]:
class Multi_Head_Attention(layers.Layer):
    def __init__(self, d_model, num_heads, **kwargs):
        super(Multi_Head_Attention, self).__init__(**kwargs)
        # Jumlah head
        self.num_heads = num_heads
        # Dimensi model
        self.d_model = d_model
        # Pastikan d_model bisa dibagi rata ke setiap head
        assert d_model % self.num_heads == 0
        # Dimensi per head
        self.depth = d_model // self.num_heads

        # Dense layer untuk Query, Key, dan Value
        self.wq = layers.Dense(d_model)
        self.wk = layers.Dense(d_model)
        self.wv = layers.Dense(d_model)

        # Dense layer untuk output akhir setelah semua head digabung
        self.dense = layers.Dense(d_model)

    # Membagi tensor ke dalam beberapa head
    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    # Scaled Dot-Product Attention
    def scaled_dot_product_attention(self, q, k, v, mask):
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

        if mask is not None:
            scaled_attention_logits += (mask * -1e9)

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, v)

        return output, attention_weights

    def call(self, v_input, k_input, q_input, mask):
        batch_size = tf.shape(q_input)[0]

        # Proyeksi linear
        q = self.wq(q_input)
        k = self.wk(k_input)
        v = self.wv(v_input)

        # Split ke beberapa head
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        # Scaled Dot Product Attention
        scaled_attention, attention_weights = self.scaled_dot_product_attention(q, k, v, mask)

        # Transpose kembali agar sesuai dimensi
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])

        # Gabungkan semua head
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))

        # Dense output akhir
        output = self.dense(concat_attention)

        return output, attention_weights


##Feed Forward Si “Transformator Andal”

In [ ]:
class Feed_Forward(layers.Layer):
    def __init__(self, d_model, dff):
        super().__init__()
        # Dense pertama dengan aktivasi ReLU (hidden layer)
        self.dense1 = layers.Dense(dff, activation='relu')
        # Dense kedua untuk mengembalikan dimensi ke d_model (output layer)
        self.dense2 = layers.Dense(d_model)

    def call(self, x):
        # Lewatkan input ke dense1 → aktivasi ReLU
        x = self.dense1(x)
        # Lewatkan hasil ke dense2 → output akhir
        x = self.dense2(x)
        return x


In [ ]:
class EncoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super(EncoderLayer, self).__init__()

        # Multi-Head Attention
        self.mha = Multi_Head_Attention(d_model, num_heads)

        # Feed Forward Network
        self.ffn = Feed_Forward(d_model, dff)

        # Layer Normalization
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        # Dropout
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, x, mask, training):
        # Multi-Head Attention + Residual Connection + LayerNorm
        attn_output, _ = self.mha(x, x, x, mask)   # Self-attention
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)    # Residual + Normalisasi

        # Feed Forward + Residual Connection + LayerNorm
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)  # Residual + Normalisasi

        return out2


##Gabungkan Seluruhnya dalam Satu “rumah” Encoder

In [ ]:
class Encoder(layers.Layer):
    def __init__(self, *, num_layers, d_model, num_heads, dff, vocab_size,
                 max_length=2048, dropout_rate=0.1):
        super().__init__()
        # Dimensi embedding model
        self.d_model = d_model
        # Jumlah lapisan encoder
        self.num_layers = num_layers

        # Token Embedding + Positional Encoding
        self.pos_embedding = Positional_Embedding(
            vocab_size=vocab_size,
            d_model=d_model,
            max_length=max_length
        )

        # Stack beberapa EncoderLayer
        self.enc_layers = [
            EncoderLayer(
                d_model=d_model,
                num_heads=num_heads,
                dff=dff,
                rate=dropout_rate
            )
            for _ in range(num_layers)
        ]

        # Dropout untuk regularisasi
        self.dropout = layers.Dropout(dropout_rate)

    # Membuat mask untuk token padding
    def create_padding_mask(self, seq):
        seq = tf.cast(tf.math.equal(seq, 0), tf.float32)
        # Bentuk mask: (batch_size, 1, 1, seq_len)
        return seq[:, tf.newaxis, tf.newaxis, :]

    def call(self, x, training):
        # Buat padding mask
        padding_mask = self.create_padding_mask(x)

        # Embedding + Positional Encoding
        x = self.pos_embedding(x)
        x = self.dropout(x, training=training)

        # Lewatkan input ke setiap EncoderLayer
        for i in range(self.num_layers):
            x = self.enc_layers[i](x, mask=padding_mask, training=training)

        return x


##Merangkum Semuanya dalam Transformer Classifier

In [ ]:
class TransformerClassifier(tf.keras.Model):
    def __init__(self, *, num_layers, d_model, num_heads, dff, vocab_size,
                 num_classes, max_length=2048, dropout_rate=0.1):
        super().__init__()

        # Encoder Block (stacked encoder layers + embedding + positional encoding)
        self.encoder = Encoder(
            num_layers=num_layers,
            d_model=d_model,
            num_heads=num_heads,
            dff=dff,
            vocab_size=vocab_size,
            max_length=max_length,
            dropout_rate=dropout_rate
        )

        # Pooling layer untuk mereduksi sequence → vektor tunggal
        self.pooling = layers.GlobalAveragePooling1D()

        # Output layer (classifier)
        # Softmax → multi-class, Sigmoid → binary
        self.classifier = layers.Dense(
            num_classes,
            activation='softmax' if num_classes > 2 else 'sigmoid'
        )

    def call(self, x, training):
        # Lewatkan input ke Encoder
        x = self.encoder(x, training=training)
        # Pooling untuk mendapatkan representasi global
        x = self.pooling(x)
        # Klasifikasi
        x = self.classifier(x)
        return x


#Latihan Melatih Model Transformer Milik Kita

##Persiapan Hyperparameter dan Dataset

In [ ]:
# Hyperparameter utama
MAX_LENGTH   = 256
BUFFER_SIZE  = 20000
BATCH_SIZE   = 64
MODEL_NAME   = 'bert-base-uncased'

NUM_LAYERS   = 2
D_MODEL      = 128
NUM_HEADS    = 4
DFF          = 256
DROPOUT_RATE = 0.1
NUM_CLASSES  = 1

# Muat dataset IMDB Reviews
(ds_train, ds_test), ds_info = tfds.load(
    'imdb_reviews',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True
)

print("✅ Dataset IMDB berhasil dimuat.")


##Inisialisasi Tokenizer

In [ ]:
# Buat tokenizer HuggingFace berdasarkan model BERT
tokenizer_hf = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ambil ukuran vocabulary aktual dari tokenizer
VOCAB_SIZE_ACTUAL = tokenizer_hf.vocab_size
print(f"✅ Tokenizer berhasil dimuat. Ukuran vocabulary: {VOCAB_SIZE_ACTUAL}")

# Pastikan pad_token_id = 0, karena model kita membuat mask berdasarkan itu
print(f"Padding token ID: {tokenizer_hf.pad_token_id}")


##Tahap Preprocessing

In [ ]:
# Fungsi preprocessing untuk encoding teks dengan HuggingFace Tokenizer
def hf_encode_py(text_tensor_py):
    # Ubah tensor menjadi string Python
    text_str = text_tensor_py.numpy().decode('utf-8')

    # Tokenisasi teks menggunakan HuggingFace AutoTokenizer
    encoded = tokenizer_hf(
        text_str,
        padding='max_length',   # padding hingga MAX_LENGTH
        truncation=True,        # potong jika lebih panjang dari MAX_LENGTH
        max_length=MAX_LENGTH,
        return_tensors="tf"     # hasil berupa tensor TensorFlow
    )

    # Ambil input_ids dengan shape (MAX_LENGTH)
    input_ids_sliced = tf.squeeze(encoded['input_ids'], axis=0)
    return input_ids_sliced


In [ ]:
@tf.function
def encode_pad(text_tensor, label_tensor):
    # Jalankan fungsi Python (hf_encode_py) di dalam graf TensorFlow
    input_ids = tf.py_function(
        func=hf_encode_py,
        inp=[text_tensor],
        Tout=tf.int32
    )

    # Tentukan shape secara manual → (MAX_LENGTH,)
    input_ids.set_shape([MAX_LENGTH])

    # Pastikan tipe data int64 (sesuai kebutuhan model)
    input_ids_int64 = tf.cast(input_ids, tf.int64)

    # Return pasangan (input_ids, label)
    return input_ids_int64, label_tensor


In [ ]:
# Terapkan preprocessing ke dataset training
ds_train_processed = ds_train.map(
    encode_pad,
    num_parallel_calls=tf.data.AUTOTUNE
)
ds_train_processed = ds_train_processed.shuffle(BUFFER_SIZE) \
                                       .batch(BATCH_SIZE) \
                                       .prefetch(tf.data.AUTOTUNE)

# Terapkan preprocessing ke dataset testing
ds_test_processed = ds_test.map(
    encode_pad,
    num_parallel_calls=tf.data.AUTOTUNE
)
ds_test_processed = ds_test_processed.batch(BATCH_SIZE) \
                                     .prefetch(tf.data.AUTOTUNE)


##Inisialisasi Model Transformer Classifier

In [ ]:
# Inisialisasi model Transformer untuk klasifikasi
transformer_classifier_model = TransformerClassifier(
    num_layers=NUM_LAYERS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dff=DFF,
    vocab_size=VOCAB_SIZE_ACTUAL,
    num_classes=NUM_CLASSES,
    max_length=MAX_LENGTH,
    dropout_rate=DROPOUT_RATE
)

# Definisi loss function dan optimizer
loss_function = tf.keras.losses.BinaryCrossentropy(from_logits=False)
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

# Compile model
transformer_classifier_model.compile(
    optimizer=optimizer,
    loss=loss_function,
    metrics=['accuracy']
)

# Bangun model dengan input dummy untuk memastikan arsitektur
dummy_input_val = tf.minimum(
    tf.cast(VOCAB_SIZE_ACTUAL - 1, tf.int64),
    1000
)  # Ambil nilai aman untuk token ID
dummy_input = tf.random.uniform(
    (BATCH_SIZE, MAX_LENGTH),
    maxval=dummy_input_val,
    dtype=tf.int64
)

# Jalankan sekali untuk membangun graph
_ = transformer_classifier_model(dummy_input, training=False)

# Ringkasan arsitektur model
transformer_classifier_model.summary()


##Waktunya Model Transformer “Belajar”

In [ ]:
# Latih Model
history = transformer_classifier_model.fit(
    ds_train_processed,
    epochs=3,
    validation_data=ds_test_processed
)


In [ ]:
# Evaluasi Model
loss, accuracy = transformer_classifier_model.evaluate(ds_test_processed)

print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")


In [ ]:
# Fungsi untuk menampilkan grafik akurasi dan loss selama training
def plot_history(history_data):
    plt.figure(figsize=(12, 4))

    # Plot Training vs Validation Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history_data.history['accuracy'], label='Training Accuracy')
    plt.plot(history_data.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Plot Training vs Validation Loss
    plt.subplot(1, 2, 2)
    plt.plot(history_data.history['loss'], label='Training Loss')
    plt.plot(history_data.history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    # Tata letak rapi
    plt.tight_layout()
    plt.show()

# Panggil fungsi untuk menampilkan grafik
plot_history(history)


##Penutup dengan Uji Coba Model Transformer

In [ ]:
# Contoh teks untuk uji prediksi
sample_texts = [
    "This movie was absolutely fantastic! The acting was superb and the plot was thrilling.",
    "I hated this film. It was boring and the storyline was predictable.",
    "An incredible masterpiece, a must-watch for everyone!",
    "What a waste of time, I would not recommend this to anyone."
]

# Fungsi prediksi sentimen
def predict_sentiment(text, hf_tokenizer_global, max_len_local):
    # Tokenisasi teks dengan HuggingFace tokenizer
    encoded = hf_tokenizer_global(
        text,
        padding='max_length',
        truncation=True,
        max_length=max_len_local,
        return_tensors="tf"
    )
    input_ids = encoded['input_ids']

    # Model mengharapkan input bertipe int64
    batched_input = tf.cast(input_ids, tf.int64)

    # Prediksi sentimen
    prediction = transformer_classifier_model.predict(batched_input, verbose=0)
    sentiment_score = prediction[0][0]
    sentiment = "Positif" if sentiment_score > 0.5 else "Negatif"

    return sentiment, sentiment_score

# Uji prediksi pada contoh teks
for text_sample in sample_texts:
    sentiment, score = predict_sentiment(text_sample, tokenizer_hf, MAX_LENGTH)
    print(f"Teks: \"{text_sample}\"")
    print(f"Prediksi: {sentiment} (Skor: {score:.4f})\n")


#Latihan Inference Object Detection

##Persiapan Dependencies dan Model Deteksi

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
from PIL import Image, ImageDraw, ImageFont, ImageColor
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import re
import requests
import cv2

In [ ]:
# Definisikan model yang ingin kita gunakan
ALL_MODELS = {
    'Faster R-CNN w/ ResNet50 v1': 'https://www.kaggle.com/models/tensorflow/faster-rcnn-resnet-v1/TensorFlow2/faster-rcnn-resnet101-v1-1024x1024/1',
    'Faster R-CNN w/ Inception ResNet v2': 'https://www.kaggle.com/models/tensorflow/faster-rcnn-inception-resnet-v2/tensorFlow2/1024x1024/1'
}

##Berhadapan dengan Label Map dari Model Deteksi

In [ ]:
# Definisikan url untuk label dari dataset
URL_LABEL = "https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/data/mscoco_label_map.pbtxt"
# Mendapatkan file dari url dan simpan dalam suatu path
PATH_TO_LABELS = tf.keras.utils.get_file(origin=URL_LABEL, fname='mscoco_label_map.pbtxt')

print(PATH_TO_LABELS)

In [ ]:
# Fungsi untuk load file label dan mapping
def load_label_map(path_label):
    label_map = {}
    with open(path_label, 'r') as f:
        label_text = f.read()

    # Mencari semua blok 'item' dalam file
    items = re.findall(r'item\s?{(.*?)}', label_text, re.DOTALL)

    # Menelusuri setiap id dan nama atau label objek pada blok item yang ditemukan
    for block in items:
        id_match = re.search(r'id:\s*(\d+)', block)
        name_match = re.search(r'display_name:\s*"([^"]+)"', block)

        # Jika kedua elemen (id dan nama) ditemukan, masukkan ke dalam label_map
        if id_match and name_match:
            item_id = int(id_match.group(1))
            display_name = name_match.group(1)
            label_map[item_id] = display_name

    print(f"Total {len(label_map)} kelas ditemukan.")

    return label_map

In [ ]:
category_map = load_label_map(PATH_TO_LABELS)

# Fungsi untuk mendownload file gambar dengan User-Agent yang sesuai kebijakan Wikimedia
def download_file(url, save_name):
    headers = {
        "User-Agent": "ColabObjectDetectionBot/1.0 (https://colab.research.google.com/; your-email@example.com)",
        "Accept": "image/webp,image/apng,image/*,*/*;q=0.8"
    }
    try:
        response = requests.get(url, headers=headers, stream=True, timeout=15)
        response.raise_for_status()

        with open(save_name, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        return save_name
    except requests.exceptions.RequestException as e:
        status_code = e.response.status_code if e.response is not None else "Unknown"
        raise Exception(f"Failed to download. Status code: {status_code}. Error: {e}")

In [ ]:
# Fungsi untuk mendownload file gambar
def download_file(url, save_name):
    # Wikimedia requires a specific User-Agent with contact info
    headers = {
        "User-Agent": "ColabObjectDetectionBot/1.0 (https://colab.research.google.com/; contact-email@example.com)",
        "Accept": "image/webp,image/apng,image/*,*/*;q=0.8"
    }
    try:
        response = requests.get(url, headers=headers, stream=True, timeout=15)
        response.raise_for_status()

        with open(save_name, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        return save_name
    except requests.exceptions.RequestException as e:
        if hasattr(e.response, 'status_code'):
            raise Exception(f"Failed to download. Status code: {e.response.status_code}. Error: {e}")
        else:
            raise Exception(f"Failed to download. Error: {e}")

In [ ]:
# Fungsi untuk load gambar
def load_image(path):
    image = cv2.imread(path)

    # Konversi gambar dari format BGR to RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Menambahkan dimensi dalam shape gambar (dibutuhkan oleh model)
    image = np.expand_dims(image, axis=0)

    return image

In [ ]:
import requests
import matplotlib.pyplot as plt

image_url = 'https://upload.wikimedia.org/wikipedia/commons/6/6b/Holiday_Plaza_-_Taxi_Stand.jpg'
save_path = 'downloaded_image.jpg'

def download_file_fixed(url, save_name):
    # Menggunakan User-Agent yang lebih unik dan menyertakan Referer
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; ColabVisionResearchBot/2.0; +https://colab.research.google.com/)",
        "Accept": "image/webp,image/apng,image/*,*/*;q=0.8",
        "Referer": "https://commons.wikimedia.org/"
    }
    try:
        # Menggunakan session untuk handling cookies jika diperlukan
        session = requests.Session()
        response = session.get(url, headers=headers, stream=True, timeout=15)
        response.raise_for_status()

        with open(save_name, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        return save_name
    except Exception as e:
        raise Exception(f"Gagal mendownload: {e}")

try:
    downloaded_image_path = download_file_fixed(image_url, save_path)
    # Pastikan load_image sudah didefinisikan sebelumnya di cell TpQ11NV7zuUk
    loaded_image = load_image(downloaded_image_path)

    print(f"Gambar berhasil didownload: {downloaded_image_path}")
    print(f"Shape gambar: {loaded_image.shape}")

    plt.imshow(loaded_image[0])
    plt.axis('off')
    plt.show()
except Exception as e:
    print(f"Error: {e}")

##Panggil Model Deteksi dan Lakukan Deteksi Objek

In [ ]:
# Pilih model deteksi yang ingin digunakan
MODEL = 'Faster R-CNN w/ Inception ResNet v2'

# Ambil URL model dari dictionary ALL_MODELS
URL_MODEL = ALL_MODELS[MODEL]

# Muat model deteksi dari TensorFlow Hub
detector = hub.load(URL_MODEL)


In [ ]:
# Hitung waktu eksekusi deteksi
start_time = time.time()

# Jalankan model deteksi pada gambar yang sudah dimuat
result_detection_tensor = detector(loaded_image)

end_time = time.time()

# Cetak durasi eksekusi
print(f"Waktu eksekusi: {end_time - start_time:.4f} detik")


##Menggambar Hasil Deteksi

In [ ]:
# Fungsi untuk menyiapkan lingkungan gambar (drawing environment)
def prepare_drawing_environment(image):
    # Konversi array NumPy menjadi gambar PIL
    image_pil = Image.fromarray(image.copy())

    # Buat objek untuk menggambar di atas gambar
    draw = ImageDraw.Draw(image_pil)

    # Ambil ukuran gambar (width, height)
    im_width, im_height = image_pil.size

    return image_pil, draw, im_width, im_height


In [ ]:
# Fungsi untuk menggambar satu bounding box dan label pada gambar
def draw_single_detection(draw, font, box, score, class_id, label_map, im_width, im_height, color_list):
    # Ambil koordinat box (ymin, xmin, ymax, xmax) dan konversi ke ukuran piksel
    ymin, xmin, ymax, xmax = box
    left, right = xmin * im_width, xmax * im_width
    top, bottom = ymin * im_height, ymax * im_height

    # Pilih warna berdasarkan class_id
    color = color_list[class_id % len(color_list)]

    # Ambil nama kelas dari label_map
    class_name = label_map.get(class_id, 'N/A')
    label_text = f"{class_name}: {int(score * 100)}%"

    # Gambar bounding box
    draw.rectangle([(left, top), (right, bottom)], outline=color, width=5)

    # Hitung ukuran teks dan gambar background teks
    text_width, text_height = font.getbbox(label_text)[2:]
    draw.rectangle(
        [(left, top - text_height - 5), (left + text_width + 5, top)],
        fill=color
    )

    # Gambar teks label di atas bounding box
    draw.text((left + 2, top - text_height - 3), label_text, font=font, fill="black")


In [ ]:
# Fungsi untuk menggambar satu bounding box dan label pada gambar
def draw_single_detection(draw, font, box, score, class_id, label_map, im_width, im_height, color_list):
    # Ambil koordinat box (ymin, xmin, ymax, xmax) dan konversi ke ukuran piksel
    ymin, xmin, ymax, xmax = box
    left, right = xmin * im_width, xmax * im_width
    top, bottom = ymin * im_height, ymax * im_height

    # Pilih warna berdasarkan class_id
    color = color_list[class_id % len(color_list)]

    # Ambil nama kelas dari label_map
    class_name = label_map.get(class_id, 'N/A')
    label_text = f"{class_name}: {int(score * 100)}%"

    # Gambar bounding box
    draw.rectangle([(left, top), (right, bottom)], outline=color, width=5)

    # Hitung ukuran teks dan gambar background teks
    text_width, text_height = font.getbbox(label_text)[2:]
    draw.rectangle(
        [(left, top - text_height - 5), (left + text_width + 5, top)],
        fill=color
    )

    # Gambar teks label di atas bounding box
    draw.text((left + 2, top - text_height - 3), label_text, font=font, fill="black")


In [ ]:
# Fungsi untuk menggambar hasil deteksi pada gambar
def draw_result_image(image, detection_result, label_map, threshold=0.4):
    # Siapkan gambar dan alat bantu gambar
    image_pil, draw, im_width, im_height = prepare_drawing_environment(image)

    # Ambil output deteksi dari hasil model
    boxes = detection_result['detection_boxes'][0]
    scores = detection_result['detection_scores'][0]
    class_labels = detection_result['detection_classes'][0].astype(int)

    # Daftar warna untuk bounding box
    color_list = [
        "red", "blue", "green", "purple", "orange", "brown", "pink", "gray",
        "cyan", "magenta", "yellow", "teal", "lime", "maroon", "navy", "olive"
    ]

    # Font untuk label teks
    font = ImageFont.truetype(
        "/usr/share/fonts/truetype/liberation/LiberationSansNarrow-Regular.ttf",
        30
    )

    # Loop semua deteksi dan gambar bounding box jika skor > threshold
    for i in range(len(scores)):
        if scores[i] > threshold:
            draw_single_detection(
                draw, font, boxes[i], scores[i], class_labels[i],
                label_map, im_width, im_height, color_list
            )

    # Kembalikan gambar hasil deteksi dalam bentuk NumPy array
    return np.array(image_pil)


In [ ]:
# Konversi hasil deteksi ke NumPy array
result_detection_np = {
    key: value.numpy() for key, value in result_detection_tensor.items()
}

# Gambar hasil deteksi pada image
detection_result_image = draw_result_image(
    loaded_image[0],       # ambil gambar pertama dari batch
    result_detection_np,   # hasil deteksi dalam format NumPy
    category_map,          # peta kategori kelas
    threshold=0.40         # hanya tampilkan deteksi dengan skor > 40%
)


In [ ]:
plt.figure(figsize=(15, 10))
plt.imshow(detection_result_image)
plt.axis('off')
plt.show()

#Latihan Fine-Tuning Object Detection

##Persiapan Dependencies

In [ ]:
!pip install tensorflow==2.19.0 keras_cv==0.9.0

In [ ]:
# Perbaikan: Pastikan tensorflow-text sinkron dengan versi tensorflow untuk menghindari NotFoundError
!pip install -q tensorflow-text==2.19.0

# Import library utama
from pathlib import Path
import yaml
import matplotlib.pyplot as plt

# Import TensorFlow dan Keras
import tensorflow as tf
from tensorflow import keras
from keras.optimizers import SGD
from keras.callbacks import Callback

# Import KerasCV untuk computer vision
import keras_cv
from keras_cv import bounding_box, visualization

# Import KaggleHub untuk akses dataset dari Kaggle
import kagglehub

##Persiapan Dataset dan Parameter

In [ ]:
# Unduh dataset Car Detection dari Kaggle menggunakan KaggleHub
path = kagglehub.dataset_download("pkdarabi/cardetection")

# Cetak lokasi folder hasil unduhan
print("Path to dataset files:", path)


In [ ]:
# Parameter utama dataset dan training
IMG_SIZE    = 640       # ukuran gambar input
MAX_BOXES   = 50        # jumlah maksimum bounding box per gambar
BATCH_SIZE  = 4         # ukuran batch untuk training

# Perbaikan: Pastikan variabel 'path' tersedia
try:
    DATASET_PATH = Path(path)
except NameError:
    # Jika variabel 'path' hilang, ambil kembali dari cache kagglehub
    path = kagglehub.dataset_download("pkdarabi/cardetection")
    DATASET_PATH = Path(path)

# Path file konfigurasi YAML dan folder dataset
PATH_DATA_YAML = DATASET_PATH.joinpath("car", "data.yaml")
TRAIN_PATH     = DATASET_PATH.joinpath("car", "train")
VAL_PATH       = DATASET_PATH.joinpath("car", "valid")
TEST_PATH      = DATASET_PATH.joinpath("car", "test")

##Mendefinisikan Kelas Objek dari data

In [ ]:
# Path ke file konfigurasi dataset (data.yaml)
PATH_DATA_YAML = DATASET_PATH.joinpath("car", "data.yaml")

# Baca file YAML untuk mendapatkan konfigurasi dataset
with open(PATH_DATA_YAML, 'r') as file:
    config_data = yaml.safe_load(file)

# Buat mapping kelas dari daftar nama di file YAML
CLASS_MAPPING = {i: nama for i, nama in enumerate(config_data['names'])}

# Cetak hasil mapping kelas
print(CLASS_MAPPING)


##Membangun Pipeline Data dengan tf.data

In [ ]:
# Fungsi untuk membaca gambar dan label YOLO, lalu konversi ke format TensorFlow
def parse_data(image_path):
    # Membaca file gambar sebagai data biner
    image = tf.io.read_file(image_path)
    # Decode data biner menjadi tensor gambar RGB
    image = tf.io.decode_jpeg(image, channels=3)
    # Resize gambar ke ukuran standar IMG_SIZE x IMG_SIZE
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])

    # Simpan dimensi gambar (height dan width)
    h = tf.cast(IMG_SIZE, tf.float32)
    w = tf.cast(IMG_SIZE, tf.float32)

    # Buat path label sesuai dengan path gambar
    label_path = tf.strings.regex_replace(image_path, "/images/", "/labels/")
    label_path = tf.strings.regex_replace(label_path, "\\.jpg$", ".txt")

    # Baca file label
    label_txt = tf.io.read_file(label_path)
    lines = tf.strings.split(label_txt, "\n")
    lines = lines[tf.strings.length(lines) > 0]  # buang baris kosong

    # Fungsi untuk menangani file label kosong
    def _empty():
        return tf.zeros([0], tf.int32), tf.zeros([0, 4], tf.float32)

    # Fungsi untuk parsing file label jika ada isinya
    def _parse():
        # Pisahkan setiap baris label menjadi angka float
        parts = tf.map_fn(
            lambda l: tf.strings.to_number(tf.strings.split(l, " "), tf.float32),
            lines,
            fn_output_signature=tf.float32,
        )
        cls = tf.cast(parts[:, 0], tf.int32)

        # Format YOLO: (cx, cy, bw, bh)
        cx, cy, bw, bh = tf.split(parts[:, 1:], 4, axis=-1)

        # Konversi ke format xyxy (x1, y1, x2, y2)
        x1 = (cx - bw / 2.0) * w
        y1 = (cy - bh / 2.0) * h
        x2 = (cx + bw / 2.0) * w
        y2 = (cy + bh / 2.0) * h

        boxes = tf.concat([x1, y1, x2, y2], axis=-1)
        return cls, boxes

    # Pilih fungsi sesuai kondisi (kosong atau ada isi)
    classes, boxes = tf.cond(tf.shape(lines)[0] == 0, _empty, _parse)

    return image, {"classes": classes, "boxes": boxes}


In [ ]:
# Fungsi untuk mengubah bounding box dari RaggedTensor ke DenseTensor
def ragged_to_dense(imgs, y):
    # Konversi bounding box ragged menjadi dense dengan jumlah maksimum box = MAX_BOXES
    y = keras_cv.bounding_box.to_dense(y, max_boxes=MAX_BOXES)
    return imgs, y


In [ ]:
# Fungsi untuk membangun pipeline tf.data.Dataset dari folder dataset
def build_dataset(path, shuffle=True):
    return (
        tf.data.Dataset
        # Ambil semua file gambar .jpg dari folder images
        .list_files(str(path / "images/*.jpg"), shuffle=shuffle)
        # Parsing gambar + label
        .map(parse_data, num_parallel_calls=tf.data.AUTOTUNE)
        # Batch ragged agar jumlah box per gambar bisa berbeda
        .ragged_batch(BATCH_SIZE, drop_remainder=True)
        # Konversi ragged ke dense (padding hingga MAX_BOXES)
        .map(ragged_to_dense, num_parallel_calls=tf.data.AUTOTUNE)
        # Prefetch untuk optimasi pipeline
        .prefetch(tf.data.AUTOTUNE)
    )


In [ ]:
# Bangun dataset untuk training, validasi, dan testing
train_dataset = build_dataset(TRAIN_PATH, shuffle=True)
val_dataset   = build_dataset(VAL_PATH, shuffle=False)
test_dataset  = build_dataset(TEST_PATH, shuffle=False)


##Visualisasi Data

In [ ]:
# Cek satu batch dari train_dataset
for images, labels in train_dataset.take(1):
    print("Batch images shape:", images.shape)
    print("Classes shape:", labels["classes"].shape)
    print("Boxes shape:", labels["boxes"].shape)


In [ ]:
# Ambil satu batch dari train_dataset
for images, labels in train_dataset.take(1):
    y_true = labels  # ground truth bounding boxes

    # Visualisasi bounding box pada galeri gambar
    keras_cv.visualization.plot_bounding_box_gallery(
        images,
        value_range=(0, 255),              # rentang nilai pixel
        bounding_box_format="xyxy",        # format koordinat bounding box
        y_true=y_true,                     # bounding box ground truth
        scale=4,                           # skala ukuran gambar
        rows=2, cols=2,                    # jumlah baris dan kolom galeri
        show=True,                         # tampilkan hasil
        font_scale=1,                      # ukuran font label
        class_mapping=CLASS_MAPPING        # mapping class id -> nama kelas
    )


##Membangun dan Melatih Model Deteksi YOLO

In [ ]:
# Definisikan backbone atau encoder model YOLO yang akan digunakan
backbone = keras_cv.models.YOLOV8Backbone.from_preset("yolo_v8_xs_backbone_coco")

In [ ]:
import keras_cv

# YOLO Prediction Decoder (NMS Layer)
prediction_decoder = keras_cv.layers.NonMaxSuppression(
    bounding_box_format="xyxy",       # format koordinat bounding box
    from_logits=True,                 # input berupa logits, bukan probabilitas
    iou_threshold=0.7,                # ambang batas IoU untuk NMS
    confidence_threshold=0.60         # ambang batas confidence score
)


In [ ]:
import keras_cv

# Perbaikan: Pastikan variabel 'backbone' tersedia
try:
    backbone_check = backbone
except NameError:
    # Jika variabel 'backbone' hilang, definisikan ulang
    backbone = keras_cv.models.YOLOV8Backbone.from_preset("yolo_v8_xs_backbone_coco")

# Definisi model YOLOv8 secara utuh
model = keras_cv.models.YOLOV8Detector(
    num_classes=len(CLASS_MAPPING),       # jumlah kelas yang akan dideteksi
    bounding_box_format="xyxy",           # format bounding box (x_min, y_min, x_max, y_max)
    backbone=backbone,                    # backbone CNN yang digunakan
    fpn_depth=1,                          # kedalaman Feature Pyramid Network
    input_shape=(IMG_SIZE, IMG_SIZE, 3),  # ukuran input gambar (H, W, C)
    prediction_decoder=prediction_decoder # decoder NMS yang sudah didefinisikan sebelumnya
)

In [ ]:
# Build model YOLO dengan input shape tertentu
model.build((None, IMG_SIZE, IMG_SIZE, 3))

# Tampilkan ringkasan arsitektur model
model.summary()


In [ ]:
# Compile model YOLO dengan optimizer SGD
optimizer = keras.optimizers.SGD(
    learning_rate=0.005,   # laju belajar
    momentum=0.9,          # momentum untuk stabilisasi update
    global_clipnorm=10     # gradient clipping
)

model.compile(
    classification_loss="binary_crossentropy",  # loss untuk klasifikasi
    box_loss="ciou",                            # loss untuk bounding box
    optimizer=optimizer
)

# Training model dengan dataset
history = model.fit(
    train_dataset,          # dataset training
    epochs=30,              # jumlah epoch
    validation_data=val_dataset,  # dataset validasi
    verbose=1               # tampilkan progress training
)


##Evaluasi Model Deteksi pada Data Test

In [ ]:
# Fungsi untuk konversi output YOLO dari dense dict → ragged dict
def to_ragged_dict(pred_dense):
    """Konversi output YOLO dict padat → ragged."""
    return {
        "boxes": tf.RaggedTensor.from_tensor(
            pred_dense["boxes"],
            padding=tf.constant([-1., -1., -1., -1.])  # padding untuk box kosong
        ),
        "confidence": tf.RaggedTensor.from_tensor(
            pred_dense["confidence"],
            padding=-1.0  # padding untuk confidence kosong
        ),
        "classes": tf.RaggedTensor.from_tensor(
            pred_dense["classes"],
            padding=-1  # padding untuk class kosong
        )
    }


In [ ]:
# Fungsi untuk visualisasi batch prediksi vs ground truth
def visualize_batch(model, dataset, bbox_fmt="xyxy"):
    # Ambil satu batch dari dataset
    images, y_true = next(iter(dataset.take(1)))

    # Lakukan prediksi dengan model
    pred_dense = model.predict(images, verbose=0)
    y_pred = to_ragged_dict(pred_dense)  # konversi ke RaggedTensor

    # Plot hasil bounding box
    visualization.plot_bounding_box_gallery(
        images,
        y_true=y_true,              # ground truth
        y_pred=y_pred,              # hasil prediksi
        bounding_box_format=bbox_fmt,
        class_mapping=CLASS_MAPPING,
        value_range=(0, 255),
        rows=2, cols=2, scale=4,
        font_scale=0.8,
        show=True,
    )

# Visualisasi batch dari dataset test
visualize_batch(model, test_dataset)


##Uji Coba Model Deteksi pada Gambar

In [ ]:
  def infer_one(model, img_path, conf_th=0.5):
    # 1) Load & resize → uint8 [0–255]
    img = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img)  # shape=(H,W,3), dtype=uint8

    # 2) Buat batch float32 [0–255] untuk model
    inp = arr.astype("float32")[None, ...]  # shape=(1,640,640,3)

    # 3) Predict (langsung pakai numpy batch)
    y_pred = model.predict(inp, verbose=0)

    # 4) Ambil array ragged → numpy
    boxes       = y_pred["boxes"][0]
    classes     = y_pred["classes"][0].astype(int)
    confidences = y_pred["confidence"][0]

    # 5) Filter by confidence
    mask = confidences > conf_th
    if not mask.any():
        print("No detections", conf_th)
        plt.imshow(arr); plt.axis("off")
        return

    filt = {
        "boxes":      boxes[mask][None, ...],
        "classes":    classes[mask][None, ...],
        "confidence": confidences[mask][None, ...],
    }

    # 6) Plot on original uint8
    keras_cv.visualization.plot_bounding_box_gallery(
        images=arr[None,...],
        y_pred=filt,
        bounding_box_format="xyxy",
        class_mapping=CLASS_MAPPING,
        value_range=(0, 255),
        rows=1, cols=1, scale=4,
    )


In [ ]:
from PIL import Image
import numpy as np

# Pastikan mengganti path di bawah dengan path gambar yang ada di folder dataset Anda
# Contoh: f"{TEST_PATH}/images/pemandangan.jpg"
try:
    test_img = list(TEST_PATH.glob('images/*.jpg'))[0]
    infer_one(model, str(test_img))
except:
    print("Gagal menemukan gambar otomatis. Silakan masukkan path manual.")
    # infer_one(model, "masukkan_path_gambar_di_sini.jpg")

##Uji Coba pada Webcam



In [ ]:
import json

# Simpan CLASS_MAPPING ke file JSON
with open('class_mapping.json', 'w') as f:
    json.dump(CLASS_MAPPING, f)

print("CLASS_MAPPING berhasil disimpan ke 'class_mapping.json'")


In [ ]:
# Simpan model YOLO ke file
model.save("yolo_sign_detection.keras")

print("✅ Model YOLO berhasil disimpan ke 'yolo_sign_detection.keras'")


##

In [ ]:
url = 'https://drive.google.com/uc?export=download&id=1awRoa3vA3s6sXnAad94MbWETdglecL_2'
data = pd.read_csv(url)

# Mengubah kolom 'date' menjadi tipe datetime dengan format YYYY-MM-DD
data['date'] = pd.to_datetime(data['date'], format='%Y-%m-%d')

# Menjadikan kolom 'date' sebagai index DataFrame
data = data.set_index('date')

data.head()

#Latihan Sequence to Sequence

##Persiapan Dependencies dan Dataset

In [ ]:
# Library untuk manipulasi data
import pandas as pd
import numpy as np
import os

# Visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

# Deep Learning
import tensorflow as tf

# Preprocessing & evaluasi dari scikit-learn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Akses dataset dari Kaggle
import kagglehub


In [ ]:
# 1) Download dataset dari Kaggle
path = kagglehub.dataset_download("shiivvvaam/bitcoin-historical-data")

# 2) Tentukan nama file dan path lengkap
file_name = "Bitcoin History.csv"
file_path = os.path.join(path, file_name)

# 3) Load dataset ke DataFrame
df = pd.read_csv(file_path)

# 4) Konversi kolom 'Date' ke tipe datetime
df["Date"] = pd.to_datetime(df["Date"], format="mixed")

# 5) Urutkan berdasarkan tanggal dan jadikan 'Date' sebagai index
df = df.sort_values("Date").set_index("Date")

# 6) Tampilkan DataFrame hasil
df


##Eksplorasi dan Preprocessing Data

In [ ]:
df.info()

In [ ]:
# Bersihkan dan konversi fitur numerik
for col in ['Price', 'Open', 'High', 'Low']:
    # 1) Pastikan semua nilai jadi string
    # 2) Hapus tanda koma (misalnya "1,234" → "1234")
    # 3) Konversi hasilnya ke float
    df[col] = df[col].astype(str).str.replace(',', '', regex=False).astype(float)

# Cek struktur DataFrame setelah konversi
df.info()


In [ ]:
# Pilih fitur yang akan dipakai
features_to_use = ['Price']
data = df[features_to_use]

# Tentukan jumlah data
n_data = len(data)

# Split: 80% train, 10% val, 10% test
train_split = int(n_data * 0.8)
val_split   = int(n_data * 0.9)

# Bagi dataset
train_data = data.iloc[:train_split]
val_data   = data.iloc[train_split:val_split]
test_data  = data.iloc[val_split:]

# Normalisasi dengan MinMaxScaler (0–1)
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data)
val_scaled   = scaler.transform(val_data)
test_scaled  = scaler.transform(test_data)


In [ ]:
WINDOW_SIZE = 300   # panjang jendela input
HORIZON     = 5     # jumlah langkah ke depan yang diprediksi
BATCH_SIZE  = 32    # ukuran batch training

def make_univariate_windows(window, horizon):
    # Encoder input → semua data kecuali horizon terakhir
    encoder_input = window[:-horizon]

    # Decoder target → horizon terakhir (nilai masa depan yang ingin diprediksi)
    decoder_target = window[-horizon:]

    # Start token → nilai terakhir dari encoder_input
    start_token = encoder_input[-1:, 0:1]

    # Decoder input → start token + target masa depan (tanpa nilai terakhir)
    decoder_input = tf.concat([start_token, decoder_target[:-1]], axis=0)

    return (encoder_input, decoder_input), decoder_target


In [ ]:
def create_univariate_dataset(series, window_size, horizon, batch_size):
    # 1) Buat dataset dari array/series
    ds = tf.data.Dataset.from_tensor_slices(series)

    # 2) Bentuk window sepanjang (window_size + horizon)
    ds = ds.window(window_size + horizon, shift=1, drop_remainder=True)

    # 3) Flatten window → jadi batch tensor
    ds = ds.flat_map(lambda w: w.batch(window_size + horizon))

    # 4) Map setiap window → (encoder_input, decoder_input), decoder_target
    ds = ds.map(lambda w: (make_univariate_windows(tf.expand_dims(w, axis=-1), horizon)))

    # 5) Batch dan prefetch untuk efisiensi training
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


In [ ]:
# Buat dataset training
train_ds = create_univariate_dataset(
    train_scaled.flatten(),  # data training dalam bentuk array 1D
    WINDOW_SIZE,             # panjang jendela input
    HORIZON,                 # jumlah langkah ke depan yang diprediksi
    BATCH_SIZE               # ukuran batch
)

# Buat dataset validasi
val_ds = create_univariate_dataset(
    val_scaled.flatten(),
    WINDOW_SIZE,
    HORIZON,
    BATCH_SIZE
)

# Buat dataset testing
test_ds = create_univariate_dataset(
    test_scaled.flatten(),
    WINDOW_SIZE,
    HORIZON,
    BATCH_SIZE
)


##Membangun Model Sequence to Sequence

In [ ]:
# Input layer untuk encoder
encoder_input = tf.keras.layers.Input(
    shape=(WINDOW_SIZE, 1),   # panjang sequence = WINDOW_SIZE, fitur = 1 (univariate)
    name="encoder_input"
)

# LSTM encoder, return_state=True agar menghasilkan hidden state & cell state
encoder_lstm = tf.keras.layers.LSTM(
    64,                       # jumlah unit LSTM
    return_state=True,
    name="encoder_lstm"
)

# Jalankan LSTM pada input → hasilkan output + state
encoder_output, state_h, state_c = encoder_lstm(encoder_input)

# Simpan state (hidden & cell) untuk dipakai decoder
encoder_states = [state_h, state_c]


In [ ]:
# Input untuk decoder → menerima sequence sepanjang HORIZON
decoder_input = tf.keras.layers.Input(
    shape=(HORIZON, 1),
    name="decoder_input"
)

# LSTM decoder → menghasilkan sequence output
# return_sequences=True agar output tiap timestep dikembalikan
# return_state=True agar state terakhir juga tersedia
decoder_lstm = tf.keras.layers.LSTM(
    64,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

# Jalankan decoder dengan initial_state dari encoder
decoder_output, state_h_dec, state_c_dec = decoder_lstm(
    decoder_input,
    initial_state=encoder_states
)

# Output layer → Dense(1) dibungkus TimeDistributed agar tiap timestep punya prediksi
output_layer = tf.keras.layers.TimeDistributed(
    tf.keras.layers.Dense(1),
    name="output"
)
output = output_layer(decoder_output)


In [ ]:
# Bangun model dengan input encoder & decoder, output prediksi
model = tf.keras.models.Model(
    inputs=[encoder_input, decoder_input],
    outputs=output
)

# Kompilasi model dengan konfigurasi training
model.compile(
    loss="mse",   # Mean Squared Error → cocok untuk regresi time series
    metrics=["mae"],  # Mean Absolute Error → metrik tambahan
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3)  # optimizer Adam
)


In [ ]:
model.summary()

In [ ]:
history = model.fit(
    train_ds,          # dataset training
    epochs=100,        # jumlah epoch (iterasi penuh)
    validation_data=val_ds  # dataset validasi untuk memantau performa
)


In [ ]:
model.evaluate(test_ds)

##Membangun Model Inference Prediksi

In [ ]:
# Ambil jumlah unit LSTM dari layer encoder
lstm_units = model.get_layer('encoder_lstm').units

# Input encoder sama seperti saat training
encoder_input = model.inputs[0]

# Ambil output dan state dari layer encoder LSTM
encoder_output, state_h_encoder, state_c_encoder = model.get_layer('encoder_lstm').output

# Simpan state encoder (hidden & cell)
encoder_states = [state_h_encoder, state_c_encoder]

# Bangun model encoder untuk inference
encoder_model = tf.keras.models.Model(
    inputs=encoder_input,
    outputs=encoder_states
)


In [ ]:
# Input state decoder (hidden & cell) dari encoder
decoder_state_input_h = tf.keras.layers.Input(
    shape=(lstm_units,), name='decoder_state_h_input'
)
decoder_state_input_c = tf.keras.layers.Input(
    shape=(lstm_units,), name='decoder_state_c_input'
)
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Input decoder untuk 1 timestep (autoregressive)
decoder_input_single = tf.keras.layers.Input(
    shape=(1, 1), name='decoder_input_single'
)

# Ambil decoder LSTM yang sudah terlatih dari model utama
decoder_lstm = model.get_layer('decoder_lstm')

# Jalankan decoder LSTM dengan initial_state dari encoder
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_input_single,
    initial_state=decoder_states_inputs
)

# Simpan state baru (akan dipakai untuk langkah berikutnya)
decoder_states = [state_h_dec, state_c_dec]

# Ambil output layer yang sudah terlatih dan hubungkan ke decoder output
output_layer = model.get_layer('output')
decoder_output_pred = output_layer(decoder_outputs)


In [ ]:
decoder_model = tf.keras.models.Model(
    inputs=[decoder_input_single] + decoder_states_inputs,
    outputs=[decoder_output_pred] + decoder_states
)


##Eksekusi Inference Prediksi

In [ ]:
def predict_sequence(input_seq, encoder_model, decoder_model, horizon):
    # 1. Dapatkan initial state dari encoder
    states_value = encoder_model.predict(input_seq)

    # 2. Ambil nilai terakhir dari input sebagai start token
    last_value = input_seq[0, -1, 0]
    target_seq = np.array([[[last_value]]])  # Shape: (1, 1, 1)

    # 3. List untuk menyimpan hasil prediksi
    output_sequence = []

    # 4. Loop autoregressive sebanyak horizon
    for _ in range(horizon):
        # Prediksi satu langkah ke depan
        output_token, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        # Simpan hasil prediksi
        output_sequence.append(output_token[0, 0, 0])

        # Update input untuk langkah berikutnya
        target_seq = output_token
        states_value = [h, c]

    return np.array(output_sequence)


In [ ]:
# Ambil satu batch dari data test untuk demonstrasi
for (encoder_in, decoder_in), decoder_target in test_ds.take(1):
    # Ambil sampel pertama dari batch
    sample_encoder_input = encoder_in[0:1]       # Shape: (1, WINDOW_SIZE, 1)
    sample_decoder_target = decoder_target[0]    # Shape: (HORIZON, 1)

# Lakukan prediksi menggunakan fungsi autoregressive yang sudah dibuat
predicted_sequence_scaled = predict_sequence(
    sample_encoder_input,
    encoder_model,
    decoder_model,
    HORIZON
)

# Inverse transform ke skala asli harga
predicted_price = scaler.inverse_transform(
    predicted_sequence_scaled.reshape(-1, 1)
)
actual_price = scaler.inverse_transform(sample_decoder_target)

# Cetak hasil prediksi vs target
print("Prediksi Harga (skala asli):", predicted_price.flatten())
print("Harga Sebenarnya (skala asli):", actual_price.flatten())


In [ ]:
# Visualisasi hasil prediksi vs harga aktual
plt.figure(figsize=(12, 6))

# Plot harga sebenarnya
plt.plot(actual_price, 'bo-', label='Harga Sebenarnya (Actual)')

# Plot harga prediksi
plt.plot(predicted_price, 'ro-', label='Harga Prediksi (Predicted)')

# Judul dan label
plt.title('Perbandingan Harga Aktual vs Prediksi untuk 5 Hari ke Depan')
plt.xlabel('Horizon (Hari ke Depan)')
plt.ylabel('Harga')

# Tambahan estetika
plt.legend()
plt.grid(True)
plt.show()


#Latihan Multivariate Sequence to Sequence

In [ ]:
data = df.copy()

##Feature Engineering

In [ ]:
data = df.copy()

# Ukuran window untuk Rolling Statistic
short_window = 7
long_window = 30

data['rolling_mean'] = data['Price'].rolling(window=short_window).mean()
data['rolling_std'] = data['Price'].rolling(window=long_window).std()

# Isi nilai NaN di awal data
data.fillna(0, inplace=True)

data

##Preprocessing Data

In [ ]:
features_to_use = ['Price', 'rolling_mean', 'rolling_std']
df = data[features_to_use]

n_data = len(df)
train_split = int(n_data * 0.8)
val_split = int(n_data * 0.9)

train_data = df.iloc[:train_split]
val_data = df.iloc[train_split:val_split]
test_data = df.iloc[val_split:]

# Scaler for all features
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data)
val_scaled = scaler.transform(val_data)

In [ ]:
WINDOW_SIZE = 300
HORIZON = 5
BATCH_SIZE = 32

def make_multivariate_windows(window, horizon):
    # Encoder input akan berisi SEMUA FITUR dari masa lalu.
    encoder_input = window[:-horizon]

    # Decoder target HANYA akan berisi fitur yang ingin kita prediksi (misalnya 'Price').
    # Ambil semua baris di horizon, tapi hanya kolom ke-0 (Price)
    decoder_target = window[-horizon:, 0:1]
    start_token = encoder_input[-1:, 0:1]

    # Gabungkan start token dengan target masa depan (tanpa nilai terakhir)
    decoder_input = tf.concat([start_token, decoder_target[:-1]], axis=0)

    # Mengembalikan tuple dari inputs dan target
    return (encoder_input, decoder_input), decoder_target

In [ ]:
def create_multivariate_dataset(series, window_size, horizon, batch_size):
    ds = tf.data.Dataset.from_tensor_slices(series)
    ds = ds.window(window_size + horizon, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda w: w.batch(window_size + horizon))

    # Removed output_signature as it's not supported in this TensorFlow version
    ds = ds.map(lambda w: make_multivariate_windows(w, horizon))

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds_multi = create_multivariate_dataset(train_scaled, WINDOW_SIZE, HORIZON, BATCH_SIZE)
val_ds_multi = create_multivariate_dataset(val_scaled, WINDOW_SIZE, HORIZON, BATCH_SIZE)
test_ds_multi = create_multivariate_dataset(test_scaled, WINDOW_SIZE, HORIZON, BATCH_SIZE)

##Membangun Model Sequence to Sequence

In [ ]:
# Encoder
encoder_input = tf.keras.layers.Input(shape=(WINDOW_SIZE, 3), name="encoder_input")
encoder_lstm = tf.keras.layers.LSTM(64, return_sequences=True, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(encoder_input)
encoder_states = [state_h, state_c]

# Decoder
decoder_input = tf.keras.layers.Input(shape=(HORIZON, 1), name="decoder_input")
decoder_lstm = tf.keras.layers.LSTM(64, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(decoder_input, initial_state=encoder_states)
decoder_dropout = tf.keras.layers.Dropout(0.5, name="decoder_dropout")(decoder_outputs)

In [ ]:
# Multi-Head Attention
attention_layer = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=lstm_units, name="multi_head_attention")
context_vector = attention_layer(query=decoder_dropout, value=encoder_outputs, key=encoder_outputs)
vector_dropout = tf.keras.layers.Dropout(0.5, name="vector_dropout")(context_vector)

concat_layer = tf.keras.layers.Concatenate(axis=-1, name="concatenate_layer")
decoder_combined_output = concat_layer([decoder_dropout, vector_dropout])

# Output Layer
output_layer = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(1), name="output")
output = output_layer(decoder_combined_output)

# Gabungkan menjadi model akhir
model_attention = tf.keras.models.Model(inputs=[encoder_input, decoder_input], outputs=output)

model_attention.summary()


In [ ]:
model_attention.compile(
    loss="mse",
    metrics=["mae"],
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3)
)

history_attention = model_attention.fit(
    train_ds_multi,
    epochs=100,
    validation_data=val_ds_multi
)

In [ ]:
model_attention.evaluate(test_ds_multi)

##Membangun Model Inference Prediksi

In [ ]:
# Dapatkan jumlah unit LSTM dan jumlah fitur dari model yang ada
lstm_units = model_attention.get_layer('encoder_lstm').units
num_features = model_attention.inputs[0].shape[-1]

# Encoder
encoder_input = model_attention.inputs[0]

# Ambil layer encoder dan outputnya
encoder_lstm_layer = model_attention.get_layer('encoder_lstm')
encoder_outputs, state_h, state_c = encoder_lstm_layer(encoder_input)

encoder_states = [state_h, state_c]

encoder_model = tf.keras.models.Model(
    inputs=encoder_input,
    outputs=[encoder_outputs] + encoder_states
)

In [ ]:
# Decoder
# Definisikan semua input yang dibutuhkan untuk satu langkah prediksi

decoder_input_single = tf.keras.layers.Input(
    shape=(1, 1),
    name='decoder_input_single'
)

decoder_state_input_h = tf.keras.layers.Input(
    shape=(lstm_units,),
    name='decoder_state_h_input'
)

decoder_state_input_c = tf.keras.layers.Input(
    shape=(lstm_units,),
    name='decoder_state_c_input'
)

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

encoder_outputs_input = tf.keras.layers.Input(
    shape=(WINDOW_SIZE, lstm_units),
    name='encoder_outputs_input'
)

# Ambil semua layer yang sudah terlatih langsung dari model asli
# Tidak perlu membuat layer baru karena sudah dikonfigurasi dengan benar

decoder_lstm = model_attention.get_layer('decoder_lstm')
attention_layer = model_attention.get_layer('multi_head_attention')
concat_layer = model_attention.get_layer('concatenate_layer')
output_layer = model_attention.get_layer('output')

decoder_dropout_layer = model_attention.get_layer('decoder_dropout')
vector_dropout_layer = model_attention.get_layer('vector_dropout')

# Rangkai ulang arsitektur untuk satu langkah prediksi

decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_input_single,
    initial_state=decoder_states_inputs
)

decoder_states_output = [
    state_h_dec,
    state_c_dec
]

d_outputs_reg = decoder_dropout_layer(
    decoder_outputs,
    training=False
)

# Hitung context vector menggunakan attention

context_vector = attention_layer(
    query=d_outputs_reg,
    value=encoder_outputs_input,
    key=encoder_outputs_input
)

c_vector_reg = vector_dropout_layer(
    context_vector,
    training=False
)

combined_output = concat_layer([
    d_outputs_reg,
    c_vector_reg
])

prediction = output_layer(combined_output)

In [ ]:
# Gabungkan menjadi decoder_model final

decoder_model = tf.keras.models.Model(
    inputs=[
        decoder_input_single,
        decoder_state_input_h,
        decoder_state_input_c,
        encoder_outputs_input
    ],
    outputs=[
        prediction,
        state_h_dec,
        state_c_dec
    ]
)

##Eksekusi Inference Prediksi

In [ ]:
def predict_sequence_attention(
    input_seq,
    encoder_model,
    decoder_model,
    horizon
):
    # 1. Dapatkan output dan state awal dari encoder
    encoder_outputs, state_h, state_c = encoder_model.predict(
        input_seq,
        verbose=0
    )

    states_value = [state_h, state_c]

    # 2. Ambil nilai terakhir dari fitur target
    # (kolom ke-0) sebagai token awal decoder
    last_value = input_seq[0, -1, 0]

    target_seq = np.array([[[last_value]]])  # Shape: (1, 1, 1)

    # 3. Simpan hasil prediksi
    output_sequence = []

    # 4. Loop autoregressive
    for _ in range(horizon):

        # Prediksi satu langkah ke depan
        output_token, h, c = decoder_model.predict(
            [
                target_seq,
                states_value[0],
                states_value[1],
                encoder_outputs
            ],
            verbose=0
        )

        output_sequence.append(output_token[0, 0, 0])

        # Output menjadi input berikutnya
        target_seq = output_token

        # Update hidden state dan cell state
        states_value = [h, c]

    return np.array(output_sequence)

In [ ]:
# Ambil satu batch dari data uji untuk demonstrasi

for (encoder_in, decoder_in), decoder_target in test_ds_multi.take(1):
    sample_encoder_input = encoder_in[0:1]      # Shape: (1, WINDOW_SIZE, num_features)
    sample_decoder_target = decoder_target[0]   # Shape: (HORIZON, 1)

# Lakukan prediksi autoregressive

predicted_sequence_scaled = predict_sequence_attention(
    sample_encoder_input,
    encoder_model,
    decoder_model,
    HORIZON
)

# Inverse transform hasil prediksi

dummy_array_pred = np.zeros(
    (len(predicted_sequence_scaled), num_features)
)

dummy_array_pred[:, 0] = predicted_sequence_scaled

predicted_price = scaler.inverse_transform(
    dummy_array_pred
)[:, 0]

# Inverse transform data aktual

dummy_array_actual = np.zeros(
    (len(sample_decoder_target), num_features)
)

dummy_array_actual[:, 0] = (
    sample_decoder_target.numpy().flatten()
)

actual_price = scaler.inverse_transform(
    dummy_array_actual
)[:, 0]

# Tampilkan hasil

print(
    f"Prediksi Harga (skala asli): "
    f"{predicted_price.flatten()}"
)

print(
    f"Harga Sebenarnya (skala asli): "
    f"{actual_price.flatten()}"
)

print()

In [ ]:
# Visualisasi Hasil
plt.figure(figsize=(12, 6))
plt.plot(actual_price, 'bo-', label='Harga Sebenarnya (Actual)')
plt.plot(predicted_price, 'ro-', label='Harga Prediksi (Predicted)')
plt.title('Perbandingan Harga Aktual vs Prediksi (Model Attention)')
plt.ylabel('Harga')
plt.xlabel('Horizon (Hari ke Depan)')
plt.legend()
plt.grid(True)
plt.show()

#Latihan Bersama TCN dan LSTM

##Persiapan Dependencies

In [ ]:
!pip install keras-tcn

In [ ]:
# ================================
# IMPORT LIBRARY
# ================================

# Mengukur waktu eksekusi program/training model
import time

# Manipulasi dan analisis data tabular (DataFrame)
import pandas as pd

# Operasi numerik dan komputasi array/matriks
import numpy as np

# Framework Deep Learning yang digunakan untuk membangun,
# melatih, dan mengevaluasi model neural network
import tensorflow as tf

# Implementasi Temporal Convolutional Network (TCN)
# Cocok untuk data time series karena mampu menangkap
# dependensi jangka panjang tanpa masalah vanishing gradient
from tcn import TCN

# Standardisasi fitur agar memiliki mean=0 dan std=1
# Penting untuk mempercepat dan menstabilkan proses training
from sklearn.preprocessing import StandardScaler

# Visualisasi data dan hasil prediksi
import matplotlib.pyplot as plt

# Mengunduh dataset langsung dari Kaggle Hub
import kagglehub

# Operasi sistem seperti membaca path/folder/file
import os


# ================================
# CATATAN PENTING
# ================================
#
# 1. NumPy (np)
#    Digunakan untuk menyimpan data dalam bentuk array
#    yang lebih efisien dibandingkan list Python.
#
# 2. Pandas (pd)
#    Cocok untuk membaca dataset CSV dan melakukan
#    preprocessing data sebelum masuk ke model.
#
# 3. StandardScaler
#    Fit scaler HANYA pada data training untuk
#    menghindari data leakage.
#
# 4. TensorFlow
#    Menjadi framework utama untuk membangun model
#    Deep Learning seperti LSTM, GRU, TCN, atau Transformer.
#
# 5. TCN
#    Alternatif LSTM yang sering lebih cepat dilatih
#    pada permasalahan forecasting time series.
#
# 6. Matplotlib
#    Digunakan untuk membandingkan hasil prediksi
#    dan data aktual melalui grafik.
#
# 7. Reproducibility
#    Sebaiknya set random seed agar hasil eksperimen
#    dapat direproduksi.
#
#    np.random.seed(42)
#    tf.random.set_seed(42)
#
# ================================

##Persiapan Dataset

In [ ]:
path = kagglehub.dataset_download("fedesoriano/wind-speed-prediction-dataset")

print("Path to dataset files:", path)

In [ ]:
# ==================================
# LOAD DAN PREPROCESS DATASET
# ==================================

# Nama file dataset
csv_file_name = 'wind_dataset.csv'

# Menggabungkan path folder dengan nama file
# sehingga kode tetap portable di berbagai sistem operasi
csv_file_path = os.path.join(path, csv_file_name)

# Membaca dataset CSV ke dalam DataFrame pandas
df = pd.read_csv(csv_file_path)

# Mengubah kolom DATE dari string menjadi tipe datetime
# format='mixed' digunakan jika format tanggal dalam dataset
# tidak sepenuhnya konsisten
df['DATE'] = pd.to_datetime(
    df['DATE'],
    format='mixed'
)

# Menjadikan kolom DATE sebagai index time series
# agar lebih mudah melakukan resampling, plotting,
# windowing, dan forecasting
df = df.set_index('DATE')

# Menampilkan 5 data pertama
df.head()

##Data Preprocessing

In [ ]:
df.info()

In [ ]:
import pandas as pd
import numpy as np

# Cek jumlah missing values sebelum perbaikan
print("Missing values sebelum imputasi:")
print(df.isnull().sum())

# Melakukan imputasi nilai yang hilang dengan forward fill (mengisi dengan nilai sebelumnya)
# Ini umum dilakukan pada data time series
df_cleaned = df.ffill().bfill()

# Verifikasi apakah masih ada NaN
print("\nMissing values setelah imputasi:")
print(df_cleaned.isnull().sum())

# Update dataset utama dengan data yang sudah bersih
df = df_cleaned

In [ ]:
# Regenerate windowed datasets using cleaned and scaled data
X_train, y_train = create_window(train_scaled, INPUT_LENGTH, OUTPUT_LENGTH, TOTAL_LENGTH)
X_val, y_val = create_window(val_scaled, INPUT_LENGTH, OUTPUT_LENGTH, TOTAL_LENGTH)
X_test, y_test = create_window(test_scaled, INPUT_LENGTH, OUTPUT_LENGTH, TOTAL_LENGTH)

print("Datasets successfully regenerated from cleaned data.")
# Verification
print(f"Any NaNs in X_train: {np.isnan(X_train).any()}")
print(f"Any NaNs in y_train: {np.isnan(y_train).any()}")

### Menjalankan Ulang Training
Sekarang silakan jalankan kembali cell training TCN (cell 23) atau LSTM. Dengan data yang sudah di-imputasi, nilai loss seharusnya tidak lagi `nan`.

In [ ]:
# ==================================
# WINDOWING CONFIGURATION
# ==================================

# Jumlah data historis (lookback window)
# yang digunakan model sebagai input
INPUT_LENGTH = 90

# Jumlah langkah waktu yang akan diprediksi
# ke masa depan (forecast horizon)
OUTPUT_LENGTH = 7

# Total panjang satu window data
# = input historis + target prediksi
TOTAL_LENGTH = INPUT_LENGTH + OUTPUT_LENGTH

In [ ]:
# ==================================
# TRAIN - VALIDATION - TEST SPLIT
# ==================================

# Hitung jumlah total data
n = len(df)

# 75% data pertama digunakan untuk training
train_df = df[0:int(n * 0.75)]

# 10% data berikutnya digunakan untuk validation
val_df = df[int(n * 0.75):int(n * 0.85)]

# 15% data terakhir digunakan untuk testing
test_df = df[int(n * 0.85):]

# Tampilkan jumlah data pada setiap subset
print(f"Train size      : {len(train_df)}")
print(f"Validation size : {len(val_df)}")
print(f"Test size       : {len(test_df)}")
print(f"Total size      : {n}")

In [ ]:
# ==================================
# DATA NORMALIZATION / STANDARDIZATION
# ==================================

# Membuat objek StandardScaler
# Rumus:
# z = (x - mean) / std
#
# Setelah transformasi:
# mean ≈ 0
# std ≈ 1

scaler = StandardScaler()

# Fit hanya pada data training
# lalu transform data training
train_scaled = scaler.fit_transform(train_df)

# Validation dan test hanya di-transform
# menggunakan statistik dari training
val_scaled = scaler.transform(val_df)
test_scaled = scaler.transform(test_df)

In [ ]:
# ==================================
# FUNGSI MEMBUAT WINDOW TIME SERIES
# ==================================

def create_window(data, input_length, output_length, total_length):
    """
    Membentuk data time series menjadi pasangan
    input (X) dan target (y) menggunakan sliding window.

    Parameters
    ----------
    data : ndarray
        Data yang sudah dinormalisasi.

    input_length : int
        Jumlah timestep historis yang digunakan sebagai input.

    output_length : int
        Jumlah timestep masa depan yang ingin diprediksi.

    total_length : int
        Panjang total satu window.
        total_length = input_length + output_length

    Returns
    -------
    X : ndarray
        Shape = (jumlah_window, input_length, jumlah_fitur)

    y : ndarray
        Shape = (jumlah_window, output_length)
    """

    X, y = [], []

    # Sliding window bergerak satu langkah setiap iterasi
    for i in range(len(data) - total_length + 1):

        # Ambil data historis sebagai input model
        X.append(
            data[i : i + input_length, :]
        )

        # Ambil nilai target masa depan
        # Kolom ke-0 diasumsikan sebagai target utama
        y.append(
            data[
                i + input_length :
                i + total_length,
                0
            ]
        )

    return np.array(X), np.array(y)


# ==================================
# MEMBUAT DATASET TRAIN, VALIDATION,
# DAN TEST DALAM BENTUK WINDOW
# ==================================

X_train, y_train = create_window(
    train_scaled,
    INPUT_LENGTH,
    OUTPUT_LENGTH,
    TOTAL_LENGTH
)

X_val, y_val = create_window(
    val_scaled,
    INPUT_LENGTH,
    OUTPUT_LENGTH,
    TOTAL_LENGTH
)

X_test, y_test = create_window(
    test_scaled,
    INPUT_LENGTH,
    OUTPUT_LENGTH,
    TOTAL_LENGTH
)

# Cek dimensi hasil windowing
print("X_train shape :", X_train.shape)
print("y_train shape :", y_train.shape)

print("X_val shape   :", X_val.shape)
print("y_val shape   :", y_val.shape)

print("X_test shape  :", X_test.shape)
print("y_test shape  :", y_test.shape)

##Membangun dan Melatih Model TCN

In [ ]:
# ==================================
# RE-INITIALIZING TCN ARCHITECTURE
# ==================================

input_shape = (X_train.shape[1], X_train.shape[2])
output_shape = y_train.shape[1]

tcn_model = tf.keras.Sequential([
    TCN(
        input_shape=input_shape,
        nb_filters=64,
        kernel_size=3,
        nb_stacks=1,
        dilations=[1, 2, 4, 8, 16],
        return_sequences=False,
        dropout_rate=0.2
    ),
    tf.keras.layers.Dense(output_shape)
])

tcn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='mae'
)

print("TCN Model Re-initialized and Compiled.")

In [ ]:
# ==================================
# TRAINING TCN MODEL WITH CLEANED DATA
# ==================================

# Reset the model to ensure a fresh start
tcn_model = tf.keras.Sequential([
    TCN(
        input_shape=(X_train.shape[1], X_train.shape[2]),
        nb_filters=64,
        kernel_size=3,
        nb_stacks=1,
        dilations=[1, 2, 4, 8, 16],
        return_sequences=False,
        dropout_rate=0.2
    ),
    tf.keras.layers.Dense(y_train.shape[1])
])

tcn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='mae'
)

epochs = 50
batch_size = 128

start_time_tcn = time.time()

history = tcn_model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, y_val),
    verbose=1
)

tcn_training_time = time.time() - start_time_tcn
print(f"Training Time TCN: {tcn_training_time:.2f} seconds")

##Membangun dan Melatih Model LSTM

In [ ]:
# ==================================
# RE-INITIALIZING LSTM ARCHITECTURE
# ==================================

lstm_model = tf.keras.Sequential([
    tf.keras.layers.LSTM(units=128, input_shape=input_shape, return_sequences=True),
    tf.keras.layers.LSTM(units=64, return_sequences=False),
    tf.keras.layers.Dense(output_shape)
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='mae'
)

print("LSTM Model Re-initialized and Compiled.")

In [ ]:
# ==================================
# TRAINING LSTM MODEL WITH CLEANED DATA
# ==================================

# Reset the model to ensure a fresh start
lstm_model = tf.keras.Sequential([
    tf.keras.layers.LSTM(units=128, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True),
    tf.keras.layers.LSTM(units=64, return_sequences=False),
    tf.keras.layers.Dense(y_train.shape[1])
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='mae'
)

start_time_lstm = time.time()

history_lstm = lstm_model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, y_val),
    verbose=1
)

lstm_training_time = time.time() - start_time_lstm
print(f"Training Time LSTM: {lstm_training_time:.2f} seconds")

##Pengujian pada Data Test

In [ ]:
# ==================================
# MENGUKUR WAKTU INFERENCE TCN
# ==================================

# Mulai menghitung waktu prediksi
start_time_tcn = time.time()

# Melakukan prediksi pada seluruh data test
predictions_tcn = tcn_model.predict(X_test)

# Total waktu inference
tcn_inference_time = time.time() - start_time_tcn


# ==================================
# MENGUKUR WAKTU INFERENCE LSTM
# ==================================

# Mulai menghitung waktu prediksi
start_time_lstm = time.time()

# Melakukan prediksi pada seluruh data test
predictions_lstm = lstm_model.predict(X_test)

# Total waktu inference
lstm_inference_time = time.time() - start_time_lstm


# ==================================
# MENAMPILKAN HASIL
# ==================================

print(f"TCN Inference Time  : {tcn_inference_time:.4f} seconds")
print(f"LSTM Inference Time : {lstm_inference_time:.4f} seconds")

In [ ]:
# Hitung loss (MAE)
tcn_loss = tcn_model.evaluate(X_test, y_test, verbose=0)
lstm_loss = lstm_model.evaluate(X_test, y_test, verbose=0)

# Ringkasan akhir
print("\n" + "=" * 45)
print("HASIL AKHIR PERBANDINGAN")
print("=" * 45)

print(f"Waktu Training TCN: {tcn_training_time:.2f} detik")
print(f"Waktu Training LSTM: {lstm_training_time:.2f} detik")

print("-" * 45)

print(f"Waktu Inference TCN di Test Set: {tcn_inference_time:.2f} detik")
print(f"Waktu Inference LSTM di Test Set: {lstm_inference_time:.2f} detik")

print("-" * 45)

print(f"Mean Absolute Error TCN di Test Set: {tcn_loss:.4f}")
print(f"Mean Absolute Error LSTM di Test Set: {lstm_loss:.4f}")

print("=" * 45)

In [ ]:
import numpy as np

# Recalculate evaluation metrics to ensure they are updated with the retrained models
tcn_loss_final = tcn_model.evaluate(X_test, y_test, verbose=0)
lstm_loss_final = lstm_model.evaluate(X_test, y_test, verbose=0)

# Redo predictions to ensure no NaNs exist in the prediction arrays
predictions_tcn = tcn_model.predict(X_test, verbose=0)
predictions_lstm = lstm_model.predict(X_test, verbose=0)

print("=== Final Evaluation Result ===")
print(f"Final TCN MAE: {tcn_loss_final:.4f}")
print(f"Final LSTM MAE: {lstm_loss_final:.4f}")

# Quick check for NaNs in predictions
if np.isnan(predictions_tcn).any() or np.isnan(predictions_lstm).any():
    print("\nWarning: NaNs still detected in predictions. Please ensure data scaling and cleaning cells were executed in order.")
else:
    print("\nSuccess: No NaNs detected in predictions.")

In [ ]:
# Re-unscale the predictions for visualization
target_index = 0
target_mean = scaler.mean_[target_index]
target_scale = scaler.scale_[target_index]

predictions_tcn_unscaled = (predictions_tcn * target_scale) + target_mean
predictions_lstm_unscaled = (predictions_lstm * target_scale) + target_mean
y_test_unscaled = (y_test * target_scale) + target_mean

# Visualization
plt.figure(figsize=(12, 6))
plt.plot(y_test_unscaled[:100, 0], label='Actual', color='blue', alpha=0.7)
plt.plot(predictions_tcn_unscaled[:100, 0], label='TCN Prediction', color='red', linestyle='--')
plt.plot(predictions_lstm_unscaled[:100, 0], label='LSTM Prediction', color='green', linestyle=':')
plt.title('Wind Speed Prediction Comparison (First 100 steps)')
plt.xlabel('Time Step')
plt.ylabel('Wind Speed')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Kolom target (kecepatan angin) berada pada indeks ke-0
target_index = 0

# Ambil nilai mean dan standar deviasi target dari scaler
target_mean = scaler.mean_[target_index]
target_scale = scaler.scale_[target_index]

# De-normalisasi hasil prediksi TCN
predictions_tcn_unscaled = (
    predictions_tcn * target_scale
) + target_mean

# De-normalisasi hasil prediksi LSTM
predictions_lstm_unscaled = (
    predictions_lstm * target_scale
) + target_mean

# De-normalisasi data aktual (y_test)
# agar berada pada skala yang sama dengan hasil prediksi
y_test_unscaled = (
    y_test * target_scale
) + target_mean


In [ ]:
# Tentukan jumlah data yang ingin ditampilkan pada grafik
plot_range = 200

# Membuat canvas visualisasi
plt.figure(figsize=(16, 8))

# Plot data aktual
plt.plot(
    y_test_unscaled[:plot_range, 0],
    label='Data Aktual',
    color='blue',
    linewidth=2
)

# Plot hasil prediksi TCN
plt.plot(
    predictions_tcn_unscaled[:plot_range, 0],
    label='Prediksi TCN',
    color='red'
)

# Plot hasil prediksi LSTM
plt.plot(
    predictions_lstm_unscaled[:plot_range, 0],
    label='Prediksi TCN 2',
    color='black'
)

# Menambahkan judul dan label sumbu
plt.title('Perbandingan Prediksi TCN vs LSTM (Data Tes)')
plt.xlabel('Time Step')
plt.ylabel('Kecepatan Angin (Satuan Asli)')

# Menampilkan legenda dan grid
plt.legend()
plt.grid(True)

# Menampilkan grafik
plt.show()

##Bereksperimen dengan TCN Buatan Sendiri

In [ ]:
# Fungsi untuk membangun satu blok konvolusi pada TCN
def convolutional_block(
    input_tensor,
    nb_filters,
    kernel_size,
    dropout_rate,
    dilation_rate
):

    # Convolution 1D dengan causal padding
    # Causal memastikan model hanya melihat data masa lalu
    x = tf.keras.layers.Conv1D(
        filters=nb_filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding='causal'
    )(input_tensor)

    # Menstabilkan distribusi aktivasi selama training
    x = tf.keras.layers.BatchNormalization()(x)

    # Menambahkan non-linearitas
    x = tf.keras.layers.Activation('relu')(x)

    # Mengurangi risiko overfitting
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    return x

In [ ]:
# Fungsi untuk membangun arsitektur utama TCN
def build_tcn(
    input_shape,
    output_shape,
    nb_filters,
    kernel_size,
    dilations,
    dropout_rate
):

    # Input layer
    inputs = tf.keras.layers.Input(shape=input_shape)
    x = inputs

    # ==================================
    # DILATED RESIDUAL BLOCKS
    # ==================================
    for d in dilations:

        # Simpan input untuk residual connection
        residual = x

        # Blok konvolusi pertama
        x_conv1 = convolutional_block(
            x,
            nb_filters,
            kernel_size,
            dropout_rate,
            dilation_rate=d
        )

        # Blok konvolusi kedua
        x_conv2 = convolutional_block(
            x_conv1,
            nb_filters,
            kernel_size,
            dropout_rate,
            dilation_rate=d
        )

        # Optional 1x1 Convolution
        # Digunakan jika jumlah channel berbeda
        if residual.shape[-1] != x_conv2.shape[-1]:
            residual = tf.keras.layers.Conv1D(
                filters=nb_filters,
                kernel_size=1
            )(residual)

        # Residual / Skip Connection
        x = tf.keras.layers.Add()([residual, x_conv2])

    # ==================================
    # MENGAMBIL TIME STEP TERAKHIR
    # ==================================
    x = tf.keras.layers.Lambda(
        lambda t: t[:, -1, :]
    )(x)

    # Output layer
    outputs = tf.keras.layers.Dense(
        output_shape,
        name='output_layer'
    )(x)

    # Membentuk model
    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs
    )

    return model


In [ ]:
# Membangun model TCN menggunakan fungsi yang telah dibuat sebelumnya
model = build_tcn(
    input_shape=input_shape,
    output_shape=output_shape,
    nb_filters=64,
    kernel_size=3,
    dilations=[1, 2, 4, 8, 16],
    dropout_rate=0.2
)

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        0.001,
        clipvalue=1.0
    ),
    loss='mae'
)

# Menampilkan ringkasan arsitektur model
model.summary()

In [ ]:
# Hyperparameter training
epochs = 100
batch_size = 128

# Mulai menghitung waktu training model TCN Manual
start_time_tcn2 = time.time()

# Proses training model
model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, y_val),
    verbose=1
)

# Menghitung total waktu training
tcn2_training_time = time.time() - start_time_tcn2

In [ ]:
# Ukur waktu inference model TCN buatan sendiri

start_time_tcn2 = time.time()

# Lakukan prediksi pada seluruh data test
predictions_tcn2 = model.predict(X_test)

# Hitung total waktu inference
tcn2_inference_time = time.time() - start_time_tcn2

In [ ]:
# Hitung loss (MAE) pada data test
tcn2_loss = model.evaluate(X_test, y_test, verbose=0)

# Tampilkan ringkasan hasil evaluasi model
print("=" * 45)
print(f"Waktu Training TCN buatan sendiri: {tcn2_training_time:.2f} detik")
print("-" * 45)
print(f"Waktu Inference TCN di Test Set: {tcn2_inference_time:.2f} detik")
print("-" * 45)
print(f"Mean Absolute Error TCN di Test Set: {tcn2_loss:.4f}")
print("=" * 45)